# 多模态与多智能体协同开发实现

---

## 课程导航

| 部分 | 内容 | 对应文件 |
|------|------|----------|
| **Part 1** | 本节定位 · 从"单点能力"到"会分工、会协同、会转人" | — |
| **Part 2** | 环境准备与模块地图 | `core/ai/*`、`core/service/*`、`core/infra/mcp_*` |
| **Part 3** | CrewAI 三角色编排 | `core/ai/crew_agents.py` |
| **Part 4** | 工具层：进程内 Tool 与 MCP 工具工厂 | `core/ai/crew_tools.py` |
| **Part 5** | 单次感知调用：意图 + 情绪 | `core/rag/multimodal.py` |
| **Part 6** | 转人工判定：四条按序规则 | `core/service/handoff.py` |
| **Part 7** | 会话回合编排与旁路记账 | `core/service/session_flow.py`、`turn_context.py` |
| **Part 8** | 人机协同：会话生命周期与坐席动作 | `core/service/session_router.py` |
| **Part 9** | MCP 工单系统：唯一外部系统 | `core/infra/mcp_client.py`、`mcp_bridge.py`、`mcp_servers/it_service/server.py` |
| **Part 10** | 实战任务 + 随堂练习（含答案） | 多模块联合 |

---

> **面向学员**：本教程面向弱基础学员，每个知识点都有详细讲解和代码注释，
> 建议按顺序逐 Cell 运行，观察每一次的输出结果。
>
> 第 35 节解决了"能听会说、能看懂图"，第 36 节解决了"知识怎么进、怎么被找到"。
> 本节课回答把能力**串成一名数字员工**的问题：
> **多个 Agent 如何分工协作？什么情况下该转人工？转人工之后又怎么运转？**

---
## Part 1：本节在架构中的位置 —— 从"单点能力"到"数字员工"
---

### 1.1 承上启下：有了感知和知识，接下来要"会干活"

第 35 节把【语音 / 截图】变成文本与结构化诊断；第 36 节把【文档】变成可检索的知识库。
但它们都是**单点能力**——像一堆零件，还不是一名"员工"。

本节要做的，是把零件组装成一名会干活、会求助的数字员工：

```
感知(35)          RAG(36)                    编排与协同(37，本节)
语音/截图 ──► 文本 ──► 解析→分块→嵌入→检索 ──► 多 Agent 分工 + 转人工 + 工单
```

### 1.2 本节的三条主线

| 主线 | 要回答的问题 | 关键代码 |
|------|--------------|----------|
| **会分工** | 谁负责回答？谁负责建单？谁负责质检？ | `crew_agents.py`（CrewAI 三角色） |
| **会判断** | 用户在问什么？情绪如何？该不该转人工？ | `multimodal.perceive_message()`、`handoff.py` |
| **会转人** | 转人工怎么通知？没坐席怎么办？工单怎么开？ | `session_router.py`、`session_flow.py`、MCP 工单系统 |

### 1.3 一条用户消息进来，会发生什么（端到端时序）

```
preflight(函数)          会话校验 / ASR / 拉历史
    │
    ▼
gate_intent(函数·router) 关闭？人工中？语音失败？关键词转人工？寒暄？
    │   ├─ 命中 → 对应终端节点（零 LLM 直接处理）
    ▼
perceive_intent(Agent·无工具)  一次 LLM：intent + emotion + 短回复
    │
    ▼
route_intent(函数·router)  非 IT 问题 → 寒暄短回复；it_issue → 继续
    │
    ▼
prepare_context(Agent·单工具 multimodal_search)  截图预分析 + 知识库检索
    │
    ▼
generate_reply(Crew·一线合成 + 按需工单 + 质检)  流式产出回复
    │
    ▼
gate_and_deliver(函数·router) 转人工判定 + 装配引用/附件 + 落库 + 推送
        ├─ 触发转人工 → trigger_handoff（切 waiting_human / 通知 / 摘要 / 建单）
        └─ 通过护栏   → 本轮结束
```

这张图就是本节的"目录"：Part 3–5 讲感知与生成，Part 6 讲护栏，Part 7 讲编排，Part 8 讲协同，Part 9 讲工单。

### 1.4 双路径开关：`CREW_ENABLED`

多 Agent 编排不是唯一解。本项目留了一个开关，两条路**对前端是同一套事件**：

| 取值 | 走哪条路 | 特点 |
|------|----------|------|
| `false`| 中台**直连 LLM 流式生成** | 逐字输出、首字快、成本低 |
| `true` | CrewAI **一线 / 工单 / 质检** 三角色编排 | Agent 自主调用检索/截图/工单工具，回复同样逐字流式 |

> 为什么保留两条路？教学上便于对比"单次调用 vs 多 Agent 协作"；工程上，
> 多 Agent 带来更强的工具自主性，但也多几次 LLM 往返，需要按场景取舍。

### 1.5 本节在五层架构中的位置

| 层 | 目录 | 本节涉及的代码 |
|---|---|---|
| 路由层 | `core/api/` | `ws_api.py`（事件推送）、`agent_api.py`（坐席操作入口） |
| 基础设施层 | `core/infra/` | **`mcp_client.py` / `mcp_bridge.py`（工单桥）**、`async_bridge.py`、`ws_manager.py` |
| **智能编排层** | **`core/ai/`** | **`crew_agents.py`（三角色）、`crew_tools.py`（工具）、`llm.py`** |
| 多模态 RAG 层 | `core/rag/` | `multimodal.py`（`perceive_message`）、`multimodal_rag.py`（检索工具底座） |
| **业务编排层** | **`core/service/`** | **`session_flow.py`（回合编排）、`handoff.py`（转人工）、`session_router.py`（会话/坐席）、`turn_context.py`（记账）** |

外部系统：`mcp_servers/it_service/server.py` —— 工单 MCP Server（**唯一**外部系统，stdio）。

### 1.6 三条贯穿全节的统一约定

1. **双路径同一出口**：无论 `CREW_ENABLED` 取值，前端收到的事件都是同一套
   `bot_stream_delta` / `bot_stream_end`——上层不需要知道底下走的是哪条路。
2. **能不用模型就不用（零 LLM 优先短路）**：关键词转人工、寒暄预过滤都排在感知之前，
   命中即短路，省掉一次 LLM 往返，用户也不用等。
3. **失败可降级、主链路不中断**：感知失败 → 保守判 `it_issue`；CrewAI 不可用 → 直连 LLM；
   无坐席 → 自动建单；MCP 不可用 → 跳过建单但仍给用户回执。

---
## Part 2：环境准备与模块地图
---

### 2.1 检查 Python 环境与依赖

In [1]:
# ============================================================
# Cell 1：环境与依赖检查
# ============================================================
import sys
import subprocess
import importlib

print("=" * 55)
print("  Python 环境检查")
print("=" * 55)
print(f"  Python 版本: {sys.version}")
print(f"  运行平台:   {sys.platform}")
print(f"  解释器路径: {sys.executable}")
print()

# 把项目根目录加入 import 路径，保证能 import core.*
sys.path.insert(0, '.')
print(f"  当前工作目录: {sys.path[0]}")
print()

required_packages = {
    'crewai': 'crewai',          # 多 Agent 编排（本节主角）
    'mcp': 'mcp',                # MCP 官方 SDK（工单外部系统）
    'openai': 'openai',          # 统一模型客户端（OpenAI 兼容协议）
    'pydantic': 'pydantic',      # 工具入参模型 / 状态模型
    'pydantic-settings': 'pydantic_settings',  # 统一配置加载
}

print("  依赖检查：")
for pkg_name, import_name in required_packages.items():
    try:
        importlib.import_module(import_name)
        print(f"     [OK] {pkg_name} 已安装")
    except ImportError:
        print(f"     [NO] {pkg_name} 未安装，正在安装...")
        try:
            r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg_name],
                               capture_output=True, text=True, timeout=180)
            print(f"     {'[OK] 安装成功' if r.returncode == 0 else '[NO] 安装失败: ' + r.stderr[:150]}")
        except Exception as e:
            print(f"     [NO] 安装失败: {e}")
print()
print("  环境检查完成。")

# ------------------------------------------------------------------
# 兼容补丁：让 asyncio.run() 在"已运行事件循环"的内核里也能用（幂等·可重入）
# ------------------------------------------------------------------
# 背景：部分 Jupyter / IDE 内核会把单元格代码放在一个**正在运行的事件循环**里执行，
# 此时直接调用 asyncio.run() 会报：
#     RuntimeError: asyncio.run() cannot be called from a running event loop
# 解决：包一层 run_async()——
#   · 当前线程【没有】运行中的事件循环：等价 asyncio.run（常规 Jupyter 行为不变）；
#   · 当前线程【已有】运行中的事件循环：把协程丢到"独立线程 + 新事件循环"执行，
#     再同步取回结果（线程内没有运行中的循环，故 asyncio.run 可正常启动）。
#
# 【为什么要写成幂等】本单元可能被复制/重复执行（例如编辑器把旧单元又贴回来）。
# 若把"原始实现"存成脚本全局变量，第二次执行会把它重绑成"上一次的补丁函数"，
# 于是在线程里调用它 → 自己调自己 → RecursionError。
# 故这里：① 用 asyncio 模块上的私有属性保存原始实现（不会被重绑覆盖）；
#          ② 加补丁标记 _mde_run_async_patched，重复执行直接跳过。
import asyncio
import concurrent.futures


def _install_asyncio_run_compat() -> bool:
    """安装兼容补丁；已安装则返回 False（不重复打补丁）。"""
    if getattr(asyncio, "_mde_run_async_patched", False):
        return False

    asyncio._mde_orig_run = asyncio.run   # 原始实现挂在模块属性上，后续不会被覆盖

    def run_async(coro, **kwargs):
        """在 notebook 中安全执行协程（兼容"已运行事件循环"的内核）。"""
        orig = asyncio._mde_orig_run      # 每次都取模块上的原始实现，确保不递归
        try:
            asyncio.get_running_loop()
        except RuntimeError:
            return orig(coro, **kwargs)   # 常规路径：无运行中的循环，直接跑
        # 兼容路径：已有运行中的循环 -> 换到独立线程的新循环里跑
        with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
            return pool.submit(lambda: orig(coro, **kwargs)).result()

    asyncio.run = run_async
    asyncio._mde_run_async_patched = True
    return True


if _install_asyncio_run_compat():
    print("已启用 run_async 兼容：asyncio.run() 现在可用于运行中的事件循环内核。")
else:
    print("run_async 兼容已启用（跳过重复补丁）。")


  Python 环境检查
  Python 版本: 3.13.7 (tags/v3.13.7:bcee1c3, Aug 14 2025, 14:15:11) [MSC v.1944 64 bit (AMD64)]
  运行平台:   win32
  解释器路径: c:\Users\13691\Documents\Code\260601-GPKJ\07-MLM\.venv\Scripts\python.exe

  当前工作目录: .

  依赖检查：
     [OK] crewai 已安装
     [OK] mcp 已安装
     [OK] openai 已安装
     [OK] pydantic 已安装
     [OK] pydantic-settings 已安装

  环境检查完成。
已启用 run_async 兼容：asyncio.run() 现在可用于运行中的事件循环内核。


> ⏳ **重要提示**：`crewai` 依赖庞大，**首次导入会明显偏慢**（离线环境下 litellm 还会
> 尝试联网拉取模型价格表而等待超时）。本节后续第一次 `import core.ai.crew_agents` 时请耐心等待。
> 导入成功后，同一内核里再次 import 会很快。
>
> 本节所有演示都**不依赖外部模型 API**：涉及"转人工判定 / 工具 schema / MCP 建单"的地方，
> 我们注入**内存 SQLite / 临时库 / 假对象**离线跑通，这正是项目 `tests/` 的做法。

### 2.2 先看模块的"说明书"：不导入也能读 docstring

每个模块开头的 `docstring` 就是它的"对外说明书"。这里有个小技巧：
用标准库 `ast` **解析源码**即可读出模块 docstring，**不需要真正 import**（快且无副作用）。

In [2]:
# ============================================================
# Cell 2：用 ast 读取各模块的"说明书"（不导入，轻量）
# ============================================================
import ast
from pathlib import Path

MODULES = [
    ("core/ai/crew_agents.py",        "CrewAI 三角色编排"),
    ("core/ai/crew_tools.py",         "进程内工具 + MCP 工具工厂"),
    ("core/service/handoff.py",       "转人工判定 / 摘要 / 工单分类"),
    ("core/service/session_flow.py",  "会话回合 Flow 编排"),
    ("core/service/session_router.py","会话生命周期与坐席动作"),
    ("core/service/turn_context.py",  "单轮记录器（旁路记账）"),
    ("core/infra/mcp_client.py",      "MCP 客户端（stdio）"),
    ("core/infra/mcp_bridge.py",      "MCP 常驻连接桥"),
    ("mcp_servers/it_service/server.py", "工单 MCP Server（外部系统）"),
]


def module_doc(path):
    """解析源码取出模块级 docstring；文件不存在返回提示。"""
    p = Path(path)
    if not p.exists():
        return "（文件不存在）"
    tree = ast.parse(p.read_text(encoding="utf-8"))
    return ast.get_docstring(tree) or "（该模块没有模块级 docstring）"


for path, label in MODULES:
    print("=" * 70)
    print(f"  {path}  ——  {label}")
    print("=" * 70)
    print(module_doc(path))
    print()

  core/ai/crew_agents.py  ——  CrewAI 三角色编排
CrewAI 多 Agent 编排（见设计文档 4.1 / 4.2）。

**接入方式（双路径开关）**：由 ``CREW_ENABLED`` 决定会话回复走哪条路——
``false``（默认）由 ``session_flow`` 走中台直连 LLM **流式生成**（逐 token 推送、首字快）；
``true`` 时由 ``session_flow`` 调本模块的 ``run_crew_turn``，由一线/工单/质检 Agent
协作产出回复（Agent 自主调用检索/截图/工单工具）。

流式：CrewAI 本身支持流式（``Crew(stream=True)`` → ``CrewStreamingOutput``，可 ``async for``
逐块迭代）。本模块用 ``await crew.kickoff_async()`` 真流式产出，逐块回调给中台推
``bot_stream_delta``。之所以要额外打一个 provider 补丁（``_strip_stream_usage``），是
**供应商侧不兼容**：硅基流动在流式响应中给每个 chunk 都回传 ``usage``，命中 CrewAI 流式
解析里「带 usage 即跳过」的分支（``_handle_streaming_completion`` 内 ``if usage: ... continue``），
导致正文 chunk 被整段丢弃、``full_response`` 恒为空。补丁对流式结果做生成器包装，凡
「有 usage 且有 choices」的 chunk 把 usage 置 None，即可让 CrewAI 正常累计正文。

三个角色：
- 一线客服 Agent：意图识别 + 多模态检索 + 截图分析，简单问题直接答复；
- 工单处理 Agent：**按需触发**（一线以 [ESCALATE: yes] 表态时才执行），
  通过 MCP 工具完成建单 / 查单 / 更新 / 密码重置 / 权限申请；
- 质检 Agent：旁路复核回复质量（不拦截、不改写，见 7.7 ⑤），默认关闭以省一次 LLM 往返。

顺序流程（Process.sequential），降低教学复杂度。
多轮上下文

### 2.3 模块地图：先有地图，再看细节

同样用 `ast` 扫一遍每个文件的顶层类 / 函数，得到一张"谁在哪、有多少东西"的地图。

In [3]:
# ============================================================
# Cell 3：模块地图（ast 扫描顶层类 / 函数）
# ============================================================
def module_map(path):
    tree = ast.parse(Path(path).read_text(encoding="utf-8"))
    items = []
    for node in tree.body:
        if isinstance(node, ast.ClassDef):
            items.append((node.lineno, "class", node.name))
        elif isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            kind = "async def" if isinstance(node, ast.AsyncFunctionDef) else "def"
            items.append((node.lineno, kind, node.name))
    return sorted(items)


for path, label in MODULES:
    items = module_map(path)
    print("=" * 70)
    print(f"  {path}   （{label}，共 {len(items)} 个顶层类/函数）")
    print("=" * 70)
    for line, kind, name in items:
        mark = "  (内部)" if name.startswith("_") else ""
        print(f"  [{kind:^9}] {name:<28} (第 {line} 行){mark}")
    print()

print("规律：以 _ 开头的是内部实现，不带 _ 的是对外接口。")

  core/ai/crew_agents.py   （CrewAI 三角色编排，共 16 个顶层类/函数）
  [  class  ] CrewUnavailable              (第 48 行)
  [  class  ] CrewTurnResult               (第 53 行)
  [   def   ] llm_configured               (第 64 行)
  [   def   ] _force_react_text_tool_calling (第 71 行)  (内部)
  [   def   ] _strip_redundant_usage       (第 103 行)  (内部)
  [   def   ] _patch_stream_usage          (第 120 行)  (内部)
  [   def   ] build_llm                    (第 151 行)
  [   def   ] _build_agents                (第 176 行)  (内部)
  [   def   ] _build_tasks                 (第 228 行)  (内部)
  [   def   ] _build_ticket_task           (第 274 行)  (内部)
  [   def   ] build_crew                   (第 296 行)
  [   def   ] _split_escalate              (第 320 行)  (内部)
  [   def   ] build_task_input             (第 336 行)
  [async def] _consume_stream              (第 371 行)  (内部)
  [async def] run_crew_turn                (第 403 行)
  [async def] _run_ticket_agent            (第 493 行)  (内部)

  core/ai/crew_tools.py   （进程内工具 + MCP 工具工厂，

---
## Part 3：CrewAI 三角色编排 —— 从"一次调用"到"会分工的团队"
---

### 3.1 需求来了：一个问题可能既要答题、又要建单、还要复核

产品经理说："数字员工不能只会回话。用户问 VPN，它要能查知识库；
查不到要能建工单；回复质量最好还能有人复核。"

最直觉的写法是把这些事全塞进**一次 LLM 调用**里。

### 3.2 v0：最直觉的写法 —— 一个巨型提示词干完所有事

In [4]:
# ============================================================
# Cell 4：v0 —— 一个巨型提示词干完所有事（反面教材，不真正执行）
# ============================================================
v0_code = """
async def answer_v0(question):
    prompt = ('你是IT客服。请：1) 如果有截图就分析截图；2) 查知识库；'
              '3) 给出答案；4) 如果解决不了就创建工单并告诉我工单号；'
              '5) 顺便检查一下你的回答质量。')
    return await llm_chat(prompt)
"""
print(v0_code)
print("痛点清单：")
print("  1. 一个提示词塞 5 件事，模型注意力被稀释，每件都做不精")
print("  2. 无法观测：不知道它到底查没查知识库、建没建单（没有工具调用记录）")
print("  3. 无法分工：建单的判断标准和答题的判断标准混在一起")
print("  4. '顺便检查质量'等于没检查，无法独立复核")
print("  5. 工单是真外部系统，必须走工具调用，不能靠模型'说出来'")


async def answer_v0(question):
    prompt = ('你是IT客服。请：1) 如果有截图就分析截图；2) 查知识库；'
              '3) 给出答案；4) 如果解决不了就创建工单并告诉我工单号；'
              '5) 顺便检查一下你的回答质量。')
    return await llm_chat(prompt)

痛点清单：
  1. 一个提示词塞 5 件事，模型注意力被稀释，每件都做不精
  2. 无法观测：不知道它到底查没查知识库、建没建单（没有工具调用记录）
  3. 无法分工：建单的判断标准和答题的判断标准混在一起
  4. '顺便检查质量'等于没检查，无法独立复核
  5. 工单是真外部系统，必须走工具调用，不能靠模型'说出来'


### 3.3 演进思路：把"一段话"拆成"角色的任务链"

正确的做法：**定义角色（Agent）→ 给每个角色配工具（Tool）→ 用任务（Task）串起来 → 顺序执行（Process.sequential）**。

```
一线IT客服代表   —— 工具：多模态检索 + 截图分析 + 工单查询
      │ 产出回复正文 + [ESCALATE: yes/no]
      ▼（按需触发，仅 yes 时）
IT工单处理专家   —— 工具：建单 / 改状态 / 密码重置 / 软件权限
      │
      ▼（可选，旁路）
服务质检专员     —— 工具：无，只给意见、不改写
```

三个角色的职责边界就是三个提示词（`role / goal / backstory`）——**分工写在提示词里，而不是塞进一段话**。

### 3.4 走读真实代码：三角色定义

> ⏳ 下面的 Cell 会首次 `import core.ai.crew_agents`，连带加载 CrewAI，**可能明显偏慢**，请耐心等待。

In [5]:
# ============================================================
# Cell 5：三角色定义源码 _build_agents
# ============================================================
import inspect
import core.ai.crew_agents as crew_agents

print("=" * 66)
print("  _build_agents —— 三个 Agent 的 role / goal / backstory / tools")
print("=" * 66)
print(inspect.getsource(crew_agents._build_agents))
print("要点：")
print("  - frontline.allow_delegation=True：一线可以'转交'，与 [ESCALATE] 机制呼应")
print("  - ticket.allow_delegation=False：工单专家只干活，不再往外转")
print("  - qa.tools=[]、max_iter=2：质检只读不写，且限制一步到底")
print("  - 三个角色共用同一个 llm 实例（build_llm()），避免重复建连")

  _build_agents —— 三个 Agent 的 role / goal / backstory / tools
def _build_agents(llm: Any, recorder: TurnRecorder) -> dict[str, Any]:
    from crewai import Agent

    frontline = Agent(
        role="一线IT客服代表",
        goal="快速理解用户问题（含截图与语音），给出准确的解决方案或正确路由",
        backstory=(
            "你是一名经验丰富的企业IT服务台客服，熟悉VPN、密码重置、软件安装、报销系统等常见问题。"
            "你态度友好，善于用简洁的语言解释技术问题。你能看懂用户上传的报错截图，"
            "并在多模态知识库中检索文本与图片方案。当问题超出你的能力范围时，"
            "你会准确地转交给工单处理专家或建议转人工。"
        ),
        tools=build_frontline_tools(recorder),
        llm=llm,
        verbose=settings.CREW_VERBOSE,
        allow_delegation=True,
        max_iter=settings.CREW_MAX_ITER,
    )

    ticket = Agent(
        role="IT工单处理专家",
        goal="根据用户需求创建、查询和更新IT服务工单",
        backstory=(
            "你是IT服务台的工单处理专家，负责将用户的问题转化为标准化工单。"
            "你熟悉工单的优先级判定、分类标准和SLA规则。你能判断什么情况下需要创建新工单、"
            "什么情况下更新已有工单。priority 取 high/medium/low，"
            "category 取 network/password/software/finance/email/general。"
   

### 3.5 任务链与顺序流程：Task + Crew(Process.sequential)

CrewAI 的 `Task.description` 就是"给这个角色的工单"。
本项目把"能否解决"的判断落成一个显式标记 `[ESCALATE: yes|no]`，
让**一线 Agent 自己表态**是否需要线下跟进——这比让中台猜要可靠得多。

In [6]:
# ============================================================
# Cell 6：任务定义与 Crew 组装源码
# ============================================================
print("=" * 66); print("  _build_tasks —— 一线任务（+ 可选质检任务）"); print("=" * 66)
print(inspect.getsource(crew_agents._build_tasks))
print("=" * 66); print("  build_crew —— 顺序流程 + 只挂载本轮有任务的 Agent"); print("=" * 66)
print(inspect.getsource(crew_agents.build_crew))
print("要点：")
print("  - memory=False：多轮上下文由【中台把历史摘要注入任务描述】实现，")
print("    刻意不用 CrewAI 内置向量记忆，避免与项目的 ChromaDB 冲突")
print("  - 只把 frontline（+可选 qa）挂进 Crew；ticket Agent 由 _run_ticket_agent 按需单独执行")

  _build_tasks —— 一线任务（+ 可选质检任务）
def _build_tasks(agents: dict[str, Any], task_input: str) -> list[Any]:
    from crewai import Task

    handle_task = Task(
        description=(
            "你正在处理一位企业员工在IT服务台的咨询。\n"
            "【对话背景】\n"
            f"{task_input}\n\n"
            "【工作要求】\n"
            "1. 先用一句话复述你理解的问题；\n"
            "2. 截图分析：若【系统预分析（截图）】已给出结果，**直接采用，不要重复调用** "
            "analyze_screenshot；仅当没有预分析结果时才调用该工具；\n"
            "3. 知识库检索：multimodal_search 是排查问题的**首要手段**。【知识库预检索参考】"
            "只是系统预先给出的线索，**不得直接当作结论**；凡涉及具体报错、配置项或操作步骤，"
            "你**必须调用 multimodal_search 亲自核实并补充**，再据此作答；仅当预检索参考已完整"
            "覆盖该问题的排查步骤、且你已核对无误时，才可跳过；\n"
            "4. 若知识库能解决，给出简明可执行的排查步骤（可用编号列表）；\n"
            "5. 若知识库无法解决或用户要求线下跟进，明确说明将转为工单/人工处理；\n"
            "6. 不要编造知识库中不存在的内容。\n"
            "【输出】先输出给用户看的中文回复正文（不要输出思考过程、不要输出 Markdown "
            "代码块围栏），随后另起一行单独输出标记：需要线下跟进/建单时输出 [ESCALATE: yes]，"
            "否则输出 [ESCALATE: no]。"
        ),
        expected_output

### 3.6 按需触发的工单 Agent：`[ESCALATE]` 标记的拆解

一线 Agent 输出正文后，会另起一行给出 `[ESCALATE: yes|no]`。
中台用一个正则把标记**从正文里剥掉**，再决定要不要跑工单 Agent——用户看到的回复里不会有这个标记。

一个关键设计：**标记缺失时按 `True` 处理**（保守：宁可多跑一次工单 Agent，也不漏建单）。

In [7]:
# ============================================================
# Cell 7：_split_escalate 源码 + 三种输入的实测
# ============================================================
print(inspect.getsource(crew_agents._split_escalate))
print()
print("实测三种情况：")
cases = [
    "这是给用户的回复正文",                       # 标记缺失 → 保守按 True
    "这是给用户的回复正文\n[ESCALATE: no]",       # 明确不需要
    "这是给用户的回复正文\n[ESCALATE: yes]",      # 明确需要线下跟进
]
for text in cases:
    body, escalate = crew_agents._split_escalate(text)
    print(f"  输入 {text!r}")
    print(f"    → 正文 {body!r}，需线下跟进 escalate={escalate}")
print()
print("结论：标记缺失按 True，是'宁可多跑、不可漏单'的工程取舍。")

def _split_escalate(text: str) -> tuple[str, bool]:
    """拆出一线输出末尾的 ``[ESCALATE: yes|no]`` 标记，返回 ``(正文, 是否需线下跟进)``。

    标记缺失时按 **True** 处理（保守：宁可多跑一次工单 Agent，也不漏建单）。
    """
    raw = (text or "").strip()
    match = _ESCALATE_RE.search(raw)
    if not match:
        return raw, True
    body = (raw[: match.start()] + raw[match.end():]).strip()
    return body, match.group(1).lower() == "yes"


实测三种情况：
  输入 '这是给用户的回复正文'
    → 正文 '这是给用户的回复正文'，需线下跟进 escalate=True
  输入 '这是给用户的回复正文\n[ESCALATE: no]'
    → 正文 '这是给用户的回复正文'，需线下跟进 escalate=False
  输入 '这是给用户的回复正文\n[ESCALATE: yes]'
    → 正文 '这是给用户的回复正文'，需线下跟进 escalate=True

结论：标记缺失按 True，是'宁可多跑、不可漏单'的工程取舍。


### 3.7 双路径开关与相关配置

`CREW_ENABLED` 决定会话回复走哪条路；`CREW_QA_ENABLED` 决定要不要跑质检（默认关，省一次 LLM 往返）。

In [8]:
# ============================================================
# Cell 8：CrewAI 相关配置与 LLM 可用性
# ============================================================
from core.config import settings

print("=" * 66)
print("  CrewAI 相关配置（来自 .env / config.py 默认值）")
print("=" * 66)
print(f"  CREW_ENABLED     = {settings.CREW_ENABLED}    （false=直连 LLM；true=CrewAI 编排）")
print(f"  CREW_QA_ENABLED  = {settings.CREW_QA_ENABLED}    （质检 Agent 旁路，默认关）")
print(f"  CREW_VERBOSE     = {settings.CREW_VERBOSE}")
print(f"  CREW_MAX_ITER    = {settings.CREW_MAX_ITER}    （单个 Agent 最大迭代步数）")
print(f"  CREW_MAX_RPM     = {settings.CREW_MAX_RPM}    （每分钟最大请求数，限流）")
print()
print(f"  LLM_MODEL        = {settings.LLM_MODEL!r}")
print(f"  LLM_BASE_URL     = {settings.LLM_BASE_URL!r}")
print(f"  llm_configured() = {crew_agents.llm_configured()}   （未配置时 build_llm 抛 CrewUnavailable）")
print()
print("  提示：CrewAI 走统一「能力四元组」的 LLM_* 配置，换供应商只改 .env，代码零改动。")

  CrewAI 相关配置（来自 .env / config.py 默认值）
  CREW_ENABLED     = False    （false=直连 LLM；true=CrewAI 编排）
  CREW_QA_ENABLED  = False    （质检 Agent 旁路，默认关）
  CREW_VERBOSE     = True
  CREW_MAX_ITER    = 5    （单个 Agent 最大迭代步数）
  CREW_MAX_RPM     = 10    （每分钟最大请求数，限流）

  LLM_MODEL        = 'Qwen/Qwen3-VL-30B-A3B-Instruct'
  LLM_BASE_URL     = 'https://api.siliconflow.cn/v1'
  llm_configured() = True   （未配置时 build_llm 抛 CrewUnavailable）

  提示：CrewAI 走统一「能力四元组」的 LLM_* 配置，换供应商只改 .env，代码零改动。


### 3.8 两个真实的"供应商兼容"坑（本节最值得学的工程经验）

这一节不是"炫技"，而是记录了两个**只有真跑起来才会遇到**的坑。
它们体现了一个重要能力：**当框架与供应商不兼容时，如何在正确的层级打补丁**。

**坑一：模型不返回原生 `tool_calls`。**
硅基流动托管的 Qwen3-VL 系列把工具调用写成 `<tool_call>{...}</tool_call>` 塞进
`reasoning_content`（`content` 为空、`tool_calls` 为 null）。CrewAI 判定"非 o1 模型即支持原生函数调用"，
于是拿到空回复并抛 `Invalid response from LLM call - None or empty`。

**坑二：流式响应里每个 chunk 都带 `usage`。**
CrewAI 流式解析里有"带 usage 即跳过"的分支，导致**正文 chunk 被整段丢弃**、`full_response` 恒为空。

两个坑的解法都是**在 provider 类上打补丁**——因为 CrewAI 内部固定实例化 `OpenAICompletion`，
子类化 `LLM` 不生效。

In [9]:
# ============================================================
# Cell 9：两个 provider 补丁的源码
# ============================================================
print("=" * 66)
print("  补丁一：_force_react_text_tool_calling —— 放弃原生函数调用，退回 ReAct 文本工具调用")
print("=" * 66)
print(inspect.getsource(crew_agents._force_react_text_tool_calling))
print("=" * 66)
print("  补丁二：_strip_redundant_usage 与 _patch_stream_usage")
print("=" * 66)
print(inspect.getsource(crew_agents._strip_redundant_usage))
print(inspect.getsource(crew_agents._patch_stream_usage))
print("要点：")
print("  - 两处都做了幂等标记（patch flag），重复调用不会叠加补丁")
print("  - 只在'有 usage 且有 choices'的 chunk 上清 usage，末尾纯统计 chunk 保留（不破坏计费口径）")
print("  - 补丁打在 provider 类上，是因为 CrewAI 内部固定实例化该子类，子类化 LLM 不生效")

  补丁一：_force_react_text_tool_calling —— 放弃原生函数调用，退回 ReAct 文本工具调用
def _force_react_text_tool_calling() -> None:
    """让 CrewAI 放弃原生函数调用、改用 ReAct 文本工具调用。

    硅基流动托管的 Qwen3-VL 系列（8B/30B/32B）不返回 OpenAI 原生 `tool_calls`，
    而是把工具调用写成 `<tool_call>{...}</tool_call>` 塞进 `reasoning_content`
    （`content` 为空、`tool_calls` 为 null）。CrewAI 依「非 o1 模型即支持原生函数
    调用」判定走原生路径，于是拿到空回复并抛
    `Invalid response from LLM call - None or empty`。

    这段文案不在 CrewAI 的「不支持原生工具」自动降级模式里，因此这里显式把 provider
    类的 `supports_function_calling` 置为 False，使 CrewAI 退回 ReAct 文本工具调用
    （工具描述写入提示词，模型以 Action/Action Input 文本触发调用）。

    注：`crewai.LLM(model=...)` 会按模型名路由到 provider 子类（本处为
    `OpenAICompletion`），子类化 LLM 不会生效，故在 provider 类上打补丁。
    """
    from crewai.llms.providers.openai.completion import OpenAICompletion

    current = OpenAICompletion.supports_function_calling
    if getattr(current, _NATIVE_TOOL_PATCH_FLAG, False):
        return

    def _no_native_tools(self) -> bool:  # noqa: ANN001
        return False



### 3.9 流式消费：逐块推送 + 尾部"扣留"

CrewAI 的流式输出（`CrewStreamingOutput`）里混着多种 chunk：工具调用、质检任务、正文……
本项目只把**一线任务（`task_index == 0`）的 TEXT 块**推给用户。

还有个细节很妙：`[ESCALATE: yes]` 最长约 16 字符，所以**尾部 32 个字符暂缓推送**，
等流结束、标记被剥离后再补推——这样标记就不会出现在用户的气泡里。

In [10]:
# ============================================================
# Cell 10：流式消费源码 _consume_stream
# ============================================================
print(inspect.getsource(crew_agents._consume_stream))
print("要领：")
print("  1. 只推 task_index==0 的 TEXT 块（质检/工具块不外泄）")
print("  2. 尾部 _ESCALATE_HOLDBACK=32 个字符暂缓，流结束剥离标记后补推")
print("  3. 返回完整原文 full，供上层再 _split_escalate 取正文")

async def _consume_stream(
    streaming: Any, on_delta: Callable[[str], Awaitable[None]]
) -> str:
    """消费 CrewAI 流式输出：逐块回调一线 Agent 的可见正文，返回完整原文。

    仅推送 ``task_index == 0``（一线 Agent 的 handle_task）的 TEXT 块；质检任务
    （task_index 1）与工具调用块不外泄。尾部 ``_ESCALATE_HOLDBACK`` 个字符暂缓推送，
    待流结束后剥离 ``[ESCALATE: …]`` 标记再补推，避免标记出现在用户气泡里。
    """
    from crewai.types.streaming import StreamChunkType

    full = ""
    sent = 0
    async for chunk in streaming:
        if getattr(chunk, "chunk_type", None) != StreamChunkType.TEXT:
            continue
        if getattr(chunk, "task_index", 0) != 0:
            continue
        piece = getattr(chunk, "content", "") or ""
        if not piece:
            continue
        full += piece
        safe_len = len(full) - _ESCALATE_HOLDBACK
        if safe_len > sent:
            await on_delta(full[sent:safe_len])
            sent = safe_len
    visible, _ = _split_escalate(full)
    if len(visible) > sent:
        await on_delta(visible[sent:])
    retu

### 3.10 小结

| 设计点 | 代码 | 为什么 |
|--------|------|--------|
| 三角色分工 | `_build_agents` | 职责写进 role/goal/backstory，各司其职 |
| 顺序流程 | `build_crew`（`Process.sequential`） | 降低教学复杂度，链路可预测 |
| 按需工单 | `[ESCALATE]` + `_run_ticket_agent` | 大部分咨询不必白跑一次工单 LLM |
| 质检旁路 | `CREW_QA_ENABLED` | 只给意见、不改写；默认关以省成本 |
| 供应商补丁 | `_force_react_text_tool_calling` / `_patch_stream_usage` | 框架与供应商不兼容时，在 provider 层修 |
| 流式扣留 | `_consume_stream` | 内部标记不出现在用户气泡里 |

---
## Part 4：工具层 —— 进程内 Tool 与 MCP 工具工厂
---

### 4.1 两类工具，两种接法

Agent 要能"动手"，就必须有工具。本项目把工具分成两类，**接入方式完全不同**：

| 类别 | 例子 | 接入方式 | 为什么 |
|------|------|----------|--------|
| **中台内部能力** | 多模态检索、截图分析 | **进程内** CrewAI `BaseTool` | 就是本项目自己的函数，没必要绕外部协议 |
| **外部系统** | 工单增删改查、密码重置 | 经 MCP Client 动态注册 | 工单是**唯一外部系统**，用 MCP 解耦 |

> 判断标准很简单：**这是"我自己的代码"还是"别人的系统"？**
> 自己的代码 → 进程内 Tool；别人的系统 → MCP。

In [11]:
# ============================================================
# Cell 11：检索工具与截图工具源码
# ============================================================
import core.ai.crew_tools as crew_tools

print(crew_tools.__doc__)
print("=" * 66); print("  MultimodalSearchTool —— 多模态检索（文查文/文查图/图查文）"); print("=" * 66)
print(inspect.getsource(crew_tools.MultimodalSearchTool))
print("=" * 66); print("  ScreenshotAnalyzeTool —— 截图分析（OCR + VLM）"); print("=" * 66)
print(inspect.getsource(crew_tools.ScreenshotAnalyzeTool))
print("要点：")
print("  - 用 pydantic ArgsModel 声明入参（Agent 靠它理解工具怎么调）")
print("  - run_sync(...) 把 async 检索投回主事件循环（Tool._run 是同步的，见 Part 9）")
print("  - screenshot 与 retrieved 都写回 recorder，供后续装配 citations/attachments")

进程内 CrewAI Tool（见设计文档 4.4）。

- 多模态检索与截图分析是**中台内部能力**，不额外起 MCP Server，以进程内 Tool 接入；
- 工单相关 Tool 由中台通过 MCP Client 动态注册（见 4.3）；
- 每次调用都写入 tool_calls（满足第十五章「全程可旁观」），并由 recorder 旁路
  记录命中结果，供会话路由装配 citations / attachments。

  MultimodalSearchTool —— 多模态检索（文查文/文查图/图查文）
class MultimodalSearchTool(BaseTool):
    """多模态知识库检索：文本与图片共用统一向量空间。"""

    name: str = "multimodal_search"
    description: str = (
        "在IT知识库中检索问题解决方案，支持文字与图片两种查询方式。"
        "用户用文字描述问题时填写 query（文查文/文查图）；"
        "用户提供报错截图时填写 image_path（图查文，找相关文本与相似截图）。"
        "返回带来源标注（文件+页码+模态）的知识片段。"
    )
    args_schema: Type[BaseModel] = MultimodalSearchArgs
    recorder: Any = Field(default=None, exclude=True)

    def _run(self, query: Optional[str] = None, image_path: Optional[str] = None) -> str:
        started = time.perf_counter()
        image_bytes = _load_image(image_path) if image_path else None
        if not query and not image_bytes:
            return "检索失败：请至少提供 query 或 image_path 之一。"
        try:
            r

### 4.2 动态注册：由 MCP 的 `inputSchema` 自动生成工具

工单工具有 8 个，如果每个都手写一个 `BaseTool` 子类，会非常啰嗦且容易与 Server 端不一致。
本项目的做法是：**读取 MCP Server 暴露的 schema，动态生成 pydantic 入参模型**，
再用一个通用 `MCPTool` 包装——Server 改了工具，中台无需改代码。

In [12]:
# ============================================================
# Cell 12：MCPTool + 动态 args_schema 源码 + 实测
# ============================================================
print(inspect.getsource(crew_tools.MCPTool))
print("=" * 66); print("  _args_model_from_schema —— 由 inputSchema 生成 pydantic 模型"); print("=" * 66)
print(inspect.getsource(crew_tools._args_model_from_schema))

print()
print("实测：用一段 create_ticket 的 inputSchema 生成入参模型")
schema = {
    "type": "object",
    "properties": {
        "title": {"type": "string", "description": "工单标题"},
        "description": {"type": "string", "description": "问题描述"},
        "priority": {"type": "string", "description": "优先级 high/medium/low"},
        "user_id": {"type": "string", "description": "用户 id"},
    },
    "required": ["title", "description"],
}
Model = crew_tools._args_model_from_schema("create_ticket", schema)
print("  生成的模型名：", Model.__name__)
for name, field in Model.model_fields.items():
    required = "必填" if field.is_required() else "可选"
    print(f"    {name:<14} {str(field.annotation):<20} {required}  # {field.description}")
print()
print("  → required 的字段类型必填、其余为 Optional；Agent 据此知道每个参数怎么填。")

class MCPTool(BaseTool):
    """把工单 MCP Server 的一个 Tool 包装成 CrewAI Tool（见 4.3）。"""

    name: str = "mcp_tool"
    description: str = ""
    args_schema: Type[BaseModel] = BaseModel
    tool_name: str = ""
    recorder: Any = Field(default=None, exclude=True)

    def _run(self, **kwargs: Any) -> str:
        started = time.perf_counter()
        try:
            text = bridge.call_tool_sync(self.tool_name, kwargs)
        except Exception as exc:  # noqa: BLE001
            logger.error("MCP 工具 %s 调用失败：%s", self.tool_name, exc)
            _log_tool_call(self.recorder, self.tool_name, kwargs, str(exc), started, status="failed")
            return f"工单系统调用失败：{exc}"
        if self.recorder is not None and self.tool_name in (
            "create_ticket", "query_ticket", "list_user_tickets", "update_ticket_status"
        ):
            self.recorder.tickets.append(text)
        _log_tool_call(self.recorder, self.tool_name, kwargs, text, started)
        return text

  _args_model_from_s

### 4.3 工具工厂：谁配哪些工具

工具不是越多越好——**按角色最小授权**：一线只能"查工单"，工单专家才能"建单/改状态"。

另外，检索工具里有一条重要规则（第 36 节讲过，这里从"工具"角度回看）：
**只在新结果分数更高时才覆盖本轮命中**，避免 Agent 自行编词发起的低分检索把高分命中冲掉。

In [13]:
# ============================================================
# Cell 13：三个工具工厂源码 + 离线实测
# ============================================================
print("=" * 66); print("  build_mcp_tools —— 按 schema 动态注册工单工具"); print("=" * 66)
print(inspect.getsource(crew_tools.build_mcp_tools))
print("=" * 66); print("  build_frontline_tools / build_ticket_tools —— 按角色最小授权"); print("=" * 66)
print(inspect.getsource(crew_tools.build_frontline_tools))
print(inspect.getsource(crew_tools.build_ticket_tools))

from core.service.turn_context import TurnRecorder

r = TurnRecorder(conversation_id="c-demo")
frontline = crew_tools.build_frontline_tools(r)
ticket = crew_tools.build_ticket_tools(r)
print()
print("离线（MCP 桥未连接）时的工具清单：")
print("  一线工具：", [t.name for t in frontline])
print("  工单工具：", [t.name for t in ticket])
print("说明：")
print("  - 检索 + 截图是进程内工具，任何时候都在")
print("  - 工单工具由 bridge.tool_schemas() 动态生成，桥未连接时为空（启动后才会出现）")
print("  - 一线只拿到 query_ticket / list_user_tickets（查询），不能建单——最小授权")

  build_mcp_tools —— 按 schema 动态注册工单工具
def build_mcp_tools(recorder: TurnRecorder, names: Optional[list[str]] = None) -> list[BaseTool]:
    """按 MCP Server 暴露的 schema 动态注册工单工具。

    Args:
        recorder: 本轮记录器。
        names: 只注册这些工具；None 表示全部。
    """
    tools: list[BaseTool] = []
    for schema in bridge.tool_schemas():
        tool_name = schema.get("name", "")
        if not tool_name:
            continue
        if names is not None and tool_name not in names:
            continue
        tools.append(
            MCPTool(
                name=tool_name,
                description=schema.get("description", "") or f"调用工单系统工具 {tool_name}",
                args_schema=_args_model_from_schema(tool_name, schema.get("inputSchema", {})),
                tool_name=tool_name,
                recorder=recorder,
            )
        )
    return tools

  build_frontline_tools / build_ticket_tools —— 按角色最小授权
def build_frontline_tools(recorder: TurnRecorder) -> list[BaseTool]:
    """一线

### 4.4 工具调用可观测：旁路记账

每次工具调用都会 `_log_tool_call(...)`：写日志 + 记到 `recorder.tool_calls`。
这样"Agent 到底调了什么、耗时多久、成功与否"全程可旁观——这就是设计文档说的"全程可旁观"。

（`TurnRecorder` 的完整源码见 Part 7。）

---
## Part 5：单次感知调用 —— 意图 + 情绪一次问清
---

### 5.1 需求：判断"来意"和"情绪"，能不能只问一次模型？

进入 RAG 之前，中台要知道两件事：
1. **来意 `intent`**：`chitchat`（寒暄）/ `off_topic`（无关）/ `it_issue`（IT 问题）；
2. **情绪 `emotion`**：`positive / neutral / negative`。

最容易想到的是"调两次模型"：一次判意图、一次判情绪。但那是**两次 LLM 往返**，又慢又贵。

本项目的做法：**一次调用同时产出两者**，连"非 IT 问题的短回复"也一起产出。

In [14]:
# ============================================================
# Cell 14：感知提示词 + perceive_message 源码
# ============================================================
import core.rag.multimodal as mm

print("=" * 66)
print("  _PERCEIVE_PROMPT —— 一次调用要模型同时给出 intent / emotion / reply")
print("=" * 66)
print(mm._PERCEIVE_PROMPT)
print()
print("=" * 66)
print("  perceive_message —— 对外接口（一次 LLM 往返）")
print("=" * 66)
print(inspect.getsource(mm.perceive_message))

  _PERCEIVE_PROMPT —— 一次调用要模型同时给出 intent / emotion / reply
你是企业IT服务台的会话前置识别助手。请对用户这句话同时完成两件事，只输出一个 JSON 对象，不要任何多余文字：

1) 判断用户来意 intent，三选一：
   - "chitchat"：寒暄、致谢、道别、闲聊等没有实质诉求的话（如“你好”“谢谢”“咋样”“在吗”）
   - "it_issue"：描述或追问 IT 相关问题（VPN、密码、邮箱、软件、网络、系统、权限、报销系统、打印机等），也包括只发截图或只描述故障现象
   - "off_topic"：与 IT 无关的其他话题（如订外卖、问天气、问你是谁）

2) 判断用户当前情绪 emotion 与 level，其中 score 表示负面情绪强度（0 表示非常平静，1 表示极度不满）。

输出格式：
{"intent": "chitchat|it_issue|off_topic", "emotion": "positive|neutral|negative",
 "level": "愤怒|焦急|平静|满意|失望|疑惑", "score": 0~1,
 "reply": "仅当 intent 不是 it_issue 时填写的一句中文短回复：chitchat 友好回应并引导用户说出遇到的IT问题；off_topic 礼貌说明你只负责IT问题并引导。intent 为 it_issue 时该字段必须是空字符串"}

判断 intent 时以“是否有需要 IT 支持的实质诉求”为准，不要因为句子短就判成 chitchat。

  perceive_message —— 对外接口（一次 LLM 往返）
async def perceive_message(text: str, audio_features: Optional[dict] = None) -> dict:
    """会话前置识别（见 6.3）：**一次 LLM 调用**同时产出「来意 intent」与「情绪 emotion」。

    - ``intent`` 决定走知识库 RAG 还是短回复，并作为置信度护栏的前置条件；
    - ``intent`` 非 ``it_issue`` 时，同一次调用直接产出回复正文 ``repl

### 5.2 失败降级：宁可当 IT 问题，也不要漏答

感知是"前置判断"，如果它挂了（未配置 / 超时 / JSON 解析失败），
不能因此不回答用户。本项目的降级策略是 **`intent` 兜底为 `it_issue`**：
照常走检索与回答——**宁可多走一遍 RAG，也不要因为识别失败漏掉真实问题**。

In [15]:
# ============================================================
# Cell 15：意图枚举与失败回退常量
# ============================================================
print("  INTENT_ENUM       =", mm.INTENT_ENUM)
print("  PERCEIVE_FALLBACK =", mm.PERCEIVE_FALLBACK)
print()
print("读法：")
print("  - intent 非法值一律归为 it_issue（保守路径）")
print("  - emotion 非法值归为 neutral")
print("  - 空文本直接返回回退值，不发起网络请求")
print()
print("配套：project 里还有 _is_chitchat() 作为'零 LLM 的寒暄预过滤'（见 Part 8），")
print("      命中寒暄的连这次感知调用都省掉。")

  INTENT_ENUM       = ('chitchat', 'it_issue', 'off_topic')
  PERCEIVE_FALLBACK = {'intent': 'it_issue', 'emotion': 'neutral', 'level': '未知', 'score': 0.0, 'reply': ''}

读法：
  - intent 非法值一律归为 it_issue（保守路径）
  - emotion 非法值归为 neutral
  - 空文本直接返回回退值，不发起网络请求

配套：project 里还有 _is_chitchat() 作为'零 LLM 的寒暄预过滤'（见 Part 8），
      命中寒暄的连这次感知调用都省掉。


---
## Part 6：转人工判定 —— 四条按序规则（本节核心）
---

### 6.1 需求来了：什么时候该把用户交给人工？

产品经理说："AI 解决不了就得转人工。但也不能动不动就转——不然人工被淹没。"

最常见、也最容易写错的版本，是**把条件用 `or` 全堆在一起**。

In [16]:
# ============================================================
# Cell 16：v0 —— 一堆 or 堆在一起（反面教材，不真正执行）
# ============================================================
v0_code = """
async def need_human_v0(conv_id, text, confidence):
    if "转人工" in text or "找客服" in text or "真人" in text:
        return True
    if confidence < 0.3:                 # 阈值拍脑袋
        return True
    if "还是不行" in text:                # 只看当前这一句
        return True
    return False
"""
print(v0_code)
print("痛点清单：")
print("  1. 条件没有顺序：'用户明确要求'优先级最高、零成本，却和低分判定混在一起")
print("  2. 阈值拍脑袋：0.3 是向量分还是精排分？不同模型的分布天差地别")
print("  3. 只看当前一句：无法识别'连续 3 轮都解决不了'这种模式")
print("  4. 没有前置条件：空知识库时必然低分，会全员误转人工")
print("  5. 寒暄也会误转：'你好'检索分天然趋近 0，被当成'低置信度'")


async def need_human_v0(conv_id, text, confidence):
    if "转人工" in text or "找客服" in text or "真人" in text:
        return True
    if confidence < 0.3:                 # 阈值拍脑袋
        return True
    if "还是不行" in text:                # 只看当前这一句
        return True
    return False

痛点清单：
  1. 条件没有顺序：'用户明确要求'优先级最高、零成本，却和低分判定混在一起
  2. 阈值拍脑袋：0.3 是向量分还是精排分？不同模型的分布天差地别
  3. 只看当前一句：无法识别'连续 3 轮都解决不了'这种模式
  4. 没有前置条件：空知识库时必然低分，会全员误转人工
  5. 寒暄也会误转：'你好'检索分天然趋近 0，被当成'低置信度'


### 6.2 演进思路：四条规则，按序判定，逐条带前置条件

| 顺序 | 规则 | 成本 | 前置条件 |
|------|------|------|----------|
| 1 | 用户明确要求（关键词命中） | **零 LLM** | 无 |
| 2 | 情绪持续负面（连续 ≥2 轮） | 一次 SQL | 最近连续用户消息均为 negative |
| 3 | 置信度低 | 用已有分数 | **有检索** + `it_issue` + `0 < 分数 < 阈值` |
| 4 | 连续未解决（连续 ≥3 轮） | 一次 SQL | 这几轮都必须是 `it_issue` |

> 顺序本身就是优化：**先用最便宜的规则短路**。规则 1 命中时，一次模型调用都不用发。

In [17]:
# ============================================================
# Cell 17：should_handoff 源码（四条按序规则）
# ============================================================
import core.service.handoff as handoff_mod

print(inspect.getsource(handoff_mod.should_handoff))
print("要点：")
print("  - 第 1 条 detect_handoff_intent 是纯字符串匹配，零 LLM，最先短路")
print("  - 第 3 条三个前置条件缺一不可：kb_nonempty + turn_confidence 非 None + it_issue")
print("  - 不再回退取'上一轮助手消息的置信度'——拿旧轮次分数判本轮，语义不成立")

async def should_handoff(
    conv_id: str,
    turn_confidence: Optional[float] = None,
    user_text: str = "",
    kb_nonempty: bool = True,
    intent: Optional[str] = None,
) -> tuple[bool, str]:
    """判断是否需要转人工，返回 (是否转人工, 原因)。判定顺序见 6.3。"""
    # 1. 用户明确要求
    if await detect_handoff_intent(user_text):
        return True, "用户明确要求转人工"

    # 2. 情绪激动（持续 2 轮以上）
    negative_rounds = await get_negative_rounds(conv_id)
    if negative_rounds >= settings.HANDOFF_NEGATIVE_ROUNDS:
        return True, f"用户情绪持续负面，已持续{negative_rounds}轮"

    # 3. AI 置信度低（见 6.3）。三个前置条件缺一不可：
    #    - 知识库非空，否则空库必然低分而误触发；
    #    - turn_confidence 非 None，即「本轮确实检索到了内容」。不再回退取上一轮助手消息的
    #      置信度——那是拿旧轮次的分数判本轮，语义不成立，且会让「本轮根本没检索」也命中；
    #    - 用户确实在问 IT 问题。优先用本轮 LLM 识别出的 intent（见 perceive_message）；
    #      未识别时（如关键词短路路径、单测）回退关键词判定，保证 LLM 不可用时不退化。
    #      寒暄/闲聊/无关提问的检索分天然趋近 0，一并判定会把这类用户误转人工
    #      （实测「你好」top1=0.0076 → 误转人工）。
    #    本轮无置信度时本条不判定，交给条件 4「连续未解决」兜底。
    it_issue = intent == "it_issue

### 6.3 规则二 / 规则四：为什么要"连续计数"，还要按 intent 过滤

规则 2 和规则 4 都是"模式识别"，靠回溯历史消息统计**连续**轮次。

规则 4 有个非常关键的细节：**只有用户在问 IT 问题的轮次才计入**。
为什么？因为寒暄/无关话题走的是短回复路径、**不写 `confidence`**，
若一并计入，闲聊几轮就会把计数垫满，导致紧随其后的真实 IT 问题被误转人工
（实测"开场欢迎语 + 2 轮寒暄"即达阈值）。

In [18]:
# ============================================================
# Cell 18：连续计数源码 get_negative_rounds / get_unresolved_rounds
# ============================================================
print("=" * 66); print("  get_negative_rounds —— 最近连续几轮情绪为负面"); print("=" * 66)
print(inspect.getsource(handoff_mod.get_negative_rounds))
print("=" * 66); print("  get_unresolved_rounds —— 最近连续几轮 IT 问题未命中知识库"); print("=" * 66)
print(inspect.getsource(handoff_mod.get_unresolved_rounds))
print("要点：")
print("  - 一旦遇到 intent != it_issue 立即 break：寒暄打断连续计数")
print("  - 只认'有配对的用户消息'的轮次：开场欢迎语没有配对用户消息，被自然排除")

  get_negative_rounds —— 最近连续几轮情绪为负面
async def get_negative_rounds(conv_id: str, window: int = 5) -> int:
    """统计最近连续几轮用户消息情绪为负面（见 6.3 触发条件 2）。"""
    rows = await db.query_all(
        "SELECT emotion FROM messages WHERE conversation_id=? AND role='user' ORDER BY id DESC LIMIT ?",
        (conv_id, window),
    )
    rounds = 0
    for row in rows:
        if row.get("emotion") == "negative":
            rounds += 1
        else:
            break
    return rounds

  get_unresolved_rounds —— 最近连续几轮 IT 问题未命中知识库
async def get_unresolved_rounds(conv_id: str, window: int = 5) -> int:
    """统计最近连续几轮「IT 问题」的 AI 回复未命中知识库（confidence 为空或 <= 0）。

    口径（见 6.3 条件 4）：**只有用户在问 IT 问题的轮次才计入**——该轮用户消息的
    ``intent`` 必须是 ``it_issue``，遇到寒暄/无关话题立即打断连续计数。

    为什么必须按 intent 过滤：寒暄/无关话题走的是短回复路径，不写 ``confidence``，
    若一并计入，闲聊几轮就会把计数垫满，导致紧随其后的真实 IT 问题被误转人工
    （实测「开场欢迎语 + 2 轮寒暄」即达阈值）。会话开场的欢迎语本身也被排除——
    它没有配对的用户消息。
    """
    rows = await db.query_all(
        "SELECT role, confidence, intent FROM me

### 6.4 规则一：关键词短路（零 LLM）

用户明确喊"转人工"是最常见的场景，也是最该**秒回**的场景。
本项目用一张关键词表做匹配——命中就短路，跳过整个感知/检索/生成链路。

但关键词表有个坑：**不能放裸词「人工」**，否则"**人工**智能"会被误命中。

In [19]:
# ============================================================
# Cell 19：关键词表 + detect_handoff_intent 源码 + 实测
# ============================================================
print("  HANDOFF_KEYWORDS（节选）：")
for kw in handoff_mod.HANDOFF_KEYWORDS:
    print(f"    {kw}", end="")
print()
print()
print(inspect.getsource(handoff_mod.detect_handoff_intent))
print()
print("实测（注意'人工智能'不会被误命中）：")
for text in ["麻烦帮我转人工", "我要找真人处理", "还是不行，帮我找人处理", "人工智能是什么", "我的VPN连不上了"]:
    hit = __import__("asyncio").run(handoff_mod.detect_handoff_intent(text))
    print(f"  {text!r:<24} → {hit}")

  HANDOFF_KEYWORDS（节选）：
    转人工    接人工    转接人工    人工客服    人工服务    找人工    要人工    叫人工    请人工    换人工    联系人工    真人    找人处理    找客服    转客服    叫工程师    找工程师    帮我找人    找个人    人工介入

async def detect_handoff_intent(text: str) -> bool:
    if not text:
        return False
    return any(keyword in text for keyword in HANDOFF_KEYWORDS)


实测（注意'人工智能'不会被误命中）：
  '麻烦帮我转人工'                → True
  '我要找真人处理'                → True
  '还是不行，帮我找人处理'            → True
  '人工智能是什么'                → False
  '我的VPN连不上了'              → False


### 6.5 实测四条规则（离线，复现 `tests/test_handoff.py`）

下面用**内存 SQLite** 复现四条规则：每条规则用一个**独立会话**（避免相互干扰），
并额外验证"寒暄不计入连续未解决"。

In [20]:
# ============================================================
# Cell 20：四条规则实测（离线，内存库）
# ============================================================
import asyncio
import os
import tempfile

from core.infra.db import Database, now_iso

# 备份原始 db，跑完恢复，避免污染后续 Cell
_orig_handoff_db = handoff_mod.db


async def _make_db():
    d = Database(os.path.join(tempfile.mkdtemp(), "nb37.db"))
    await d.init()
    # conversations.user_id 有外键约束，先建两个测试用户
    for uid in ("u-1", "u-2"):
        await d.execute(
            """INSERT INTO users (id, username, password_hash, display_name, email,
               department, role, status, created_at)
               VALUES (?,?,?,?,'','','user','active',?)""",
            (uid, uid, "x", uid, now_iso()),
        )
    return d


test_db = asyncio.run(_make_db())
handoff_mod.db = test_db
threshold = settings.HANDOFF_MIN_CONFIDENCE

print("=" * 66)
print("  规则 1：用户明确要求（关键词，零 LLM）")
print("=" * 66)
c1 = asyncio.run(test_db.create_conversation("u-1"))
print("  ->", asyncio.run(handoff_mod.should_handoff(c1, user_text="麻烦帮我转人工")))

print()
print("=" * 66)
print("  规则 2：情绪持续负面（连续 ≥ HANDOFF_NEGATIVE_ROUNDS）")
print("=" * 66)
c2 = asyncio.run(test_db.create_conversation("u-1"))
asyncio.run(test_db.add_message(c2, "user", "怎么还是不行", emotion="negative"))
asyncio.run(test_db.add_message(c2, "assistant", "请再试一次", confidence=0.9))
asyncio.run(test_db.add_message(c2, "user", "你们这系统真差劲", emotion="negative"))
print("  ->", asyncio.run(handoff_mod.should_handoff(
    c2, user_text="你们这系统真差劲", kb_nonempty=False)))

print()
print("=" * 66)
print(f"  规则 3：置信度低（0 < 分数 < {threshold}，且为 it_issue）")
print("=" * 66)
c3 = asyncio.run(test_db.create_conversation("u-1"))
print("  低分 0.05 ->", asyncio.run(handoff_mod.should_handoff(
    c3, turn_confidence=0.05, user_text="报销系统打不开", kb_nonempty=True, intent="it_issue")))
print("  高分 0.60 ->", asyncio.run(handoff_mod.should_handoff(
    c3, turn_confidence=0.60, user_text="报销系统打不开", kb_nonempty=True, intent="it_issue")))
print("  寒暄 + 低分 ->", asyncio.run(handoff_mod.should_handoff(
    c3, turn_confidence=0.01, user_text="你好", kb_nonempty=True, intent="chitchat")))
print("  （寒暄被前置条件 intent==it_issue 挡住，不误转）")

print()
print("=" * 66)
print(f"  规则 4：连续未解决（连续 ≥ {settings.HANDOFF_UNRESOLVED_ROUNDS} 轮）")
print("=" * 66)
c4 = asyncio.run(test_db.create_conversation("u-1"))
for i in range(settings.HANDOFF_UNRESOLVED_ROUNDS):
    asyncio.run(test_db.add_message(c4, "user", f"问题{i}", emotion="neutral", intent="it_issue"))
    asyncio.run(test_db.add_message(c4, "assistant", "抱歉，暂未找到方案"))
print("  ->", asyncio.run(handoff_mod.should_handoff(
    c4, turn_confidence=None, user_text="还是不行", kb_nonempty=True, intent="it_issue")))

print()
print("=" * 66)
print("  反例：寒暄不计入'连续未解决'")
print("=" * 66)
c5 = asyncio.run(test_db.create_conversation("u-2"))
asyncio.run(test_db.add_message(c5, "assistant", "您好，我是IT服务台数字员工"))
for text in ("你好", "嘿嘿", "谢谢"):
    asyncio.run(test_db.add_message(c5, "user", text, emotion="neutral", intent="chitchat"))
    asyncio.run(test_db.add_message(c5, "assistant", "（寒暄短回复）"))
print(f"  get_unresolved_rounds = {asyncio.run(handoff_mod.get_unresolved_rounds(c5))}  （应为 0）")
print("  紧接着的真实 IT 问题 ->", asyncio.run(handoff_mod.should_handoff(
    c5, turn_confidence=0.70, user_text="VPN连不上", kb_nonempty=True, intent="it_issue")))

# 恢复原始 db
handoff_mod.db = _orig_handoff_db
print()
print("  已恢复 handoff.db。")

  规则 1：用户明确要求（关键词，零 LLM）
  -> (True, '用户明确要求转人工')

  规则 2：情绪持续负面（连续 ≥ HANDOFF_NEGATIVE_ROUNDS）
  -> (True, '用户情绪持续负面，已持续2轮')

  规则 3：置信度低（0 < 分数 < 0.15，且为 it_issue）
  低分 0.05 -> (True, '知识库检索置信度0.05，低于阈值0.15')
  高分 0.60 -> (False, '')
  寒暄 + 低分 -> (False, '')
  （寒暄被前置条件 intent==it_issue 挡住，不误转）

  规则 4：连续未解决（连续 ≥ 3 轮）
  -> (True, '连续3轮未解决')

  反例：寒暄不计入'连续未解决'
  get_unresolved_rounds = 0  （应为 0）
  紧接着的真实 IT 问题 -> (False, '')

  已恢复 handoff.db。


### 6.6 交接摘要：坐席不用把问题再问一遍

触发转人工后，要给坐席一份**交接摘要**（问题是什么 / AI 做了什么 / 卡在哪）。
这里的工程顺序很重要：**先切状态并通知用户，再生成摘要**——摘要有一次 LLM 调用（数秒），
不能让用户在"转接中"干等。

In [21]:
# ============================================================
# Cell 21：generate_handoff_summary 源码
# ============================================================
print(inspect.getsource(handoff_mod.generate_handoff_summary))
print("要点：")
print("  - 摘要有 LLM 时用 LLM 生成；无 LLM 时用'最后一条用户消息 + 消息数'兜底")
print("  - 同时给出 emotion_trend（负面轮次 ≥2 记 negative），供坐席端提醒")

async def generate_handoff_summary(conv_id: str, reason: str = "") -> dict:
    """生成转人工摘要（见 6.4）：坐席无需重新问一遍。"""
    messages = await db.get_messages(conv_id, limit=100)
    conversation = await db.get_conversation(conv_id) or {}

    transcript_lines: list[str] = []
    for msg in messages[-30:]:
        role_label = {"user": "用户", "assistant": "AI客服", "agent": "人工客服", "system": "系统"}.get(
            msg.get("role", ""), msg.get("role", "")
        )
        transcript_lines.append(f"{role_label}：{msg.get('content', '')}")

    summary = ""
    if llm_available() and transcript_lines:
        prompt = (
            "请用不超过 3 句话总结这次IT服务台客服对话：\n"
            "1) 用户的问题是什么；2) AI 已经做了什么；3) 当前卡在哪里。\n"
            "只输出总结正文，不要任何前后缀。\n\n对话记录：\n" + "\n".join(transcript_lines)
        )
        result = await chat_json(
            "请直接输出 JSON：{\"summary\": \"...\"}。\n\n" + prompt,
            system="你是客服值班主管，擅长用简洁语言做对话交接摘要。",
        )
        if result and result.get("summary"):
            su

### 6.7 自动建单与去重：工单是"唯一外部系统"

建单要先分类（标题/描述/优先级/分类），再调用 MCP 的 `create_ticket`。
这里有两个很关键的工程细节：

1. **建单必须写入 `tool_calls`**：`list_conversation_tickets()` 靠它做去重——
   否则关闭会话时会认为"无关联工单"而**重复建单**。
2. **失败也要落一条 `failed` 审计**：失败留痕，与工具路径的口径一致。

优先级/分类的枚举与知识库分类共用：`network/password/software/finance/email/general`。

In [22]:
# ============================================================
# Cell 22：工单分类 / 建单 / 去重源码
# ============================================================
print("=" * 66); print("  _classify_ticket —— LLM 判定标题/描述/优先级/分类"); print("=" * 66)
print(inspect.getsource(handoff_mod._classify_ticket))
print("=" * 66); print("  create_ticket_for_conversation —— 自动建单（含审计落库）"); print("=" * 66)
print(inspect.getsource(handoff_mod.create_ticket_for_conversation))
print("=" * 66); print("  list_conversation_tickets —— 从工具日志提取已建工单号（去重）"); print("=" * 66)
print(inspect.getsource(handoff_mod.list_conversation_tickets))
print("枚举：")
print("  CATEGORY_ENUM =", handoff_mod.CATEGORY_ENUM)
print("  PRIORITY_ENUM =", handoff_mod.PRIORITY_ENUM)

  _classify_ticket —— LLM 判定标题/描述/优先级/分类
async def _classify_ticket(conv_id: str, extra: str = "") -> dict:
    """用 LLM 判定工单标题/描述/优先级/分类（规则见 5.5）。"""
    messages = await db.get_messages(conv_id, limit=40)
    transcript = "\n".join(
        f"{'用户' if m.get('role') == 'user' else 'AI/坐席'}：{m.get('content', '')}" for m in messages[-20:]
    )
    default = {
        "title": "IT问题跟进",
        "description": f"会话 {conv_id} 问题未解决，需线下跟进。{extra}".strip(),
        "priority": "medium",
        "category": "general",
    }
    if not llm_available() or not transcript:
        return default

    prompt = (
        "根据下面的IT服务台对话，生成一个工单。严格只输出 JSON，字段：\n"
        '{"title": "不超过20字的标题", "description": "问题描述与已做排查", '
        '"priority": "high|medium|low", "category": "network|password|software|finance|email|general"}\n'
        "判定标准：多人受影响/核心系统不可用/明确加急=high；单人问题但有绕过方式=medium；"
        "咨询类不阻塞=low。\n\n对话：\n" + transcript
    )
    data = await chat_json(prompt, system="你是IT服务台工单受理员，输出必须是合法 JSON

---
## Part 7：会话回合编排 —— CrewAI Flow 与旁路记账
---

### 7.1 Flow 的定位：把节点编排成一条 DAG

`session_flow.py` 用 CrewAI Flow 重新表达"一条用户消息进来，一条回复（或转人工）出去"。
节点必须定义在 `SessionTurnFlow` 类体内（FlowMeta 靠类命名空间扫描装饰器）。

节点按类型分三类，**阶梯式展示 Agent 能力**：

| 类型 | 例子 | 说明 |
|------|------|------|
| ① 函数节点 | `preflight` / `gate_intent` / `route_intent` / `gate_and_deliver` | 零 LLM 的闸门、准备、装配 |
| ② Agent 节点（无工具） | `perceive_intent` | 单 Agent 一次结构化 LLM 往返 |
| ② Agent 节点（单工具） | `prepare_context` | Agent 自主调用 `multimodal_search` |
| ③ Crew 节点 | `generate_reply` | 多 Agent + 多工具 |

状态由一个 pydantic 模型 `TurnState` 承载，节点间通过 `self.state` 传递。

In [23]:
# ============================================================
# Cell 23：TurnState —— Flow 的状态载体
# ============================================================
import core.service.session_flow as session_flow

print("=" * 66)
print("  TurnState —— 输入 + 各节点产出（节点间靠它传递）")
print("=" * 66)
print(inspect.getsource(session_flow.TurnState))
print("读法：")
print("  输入区：conv_id / user_id / content / modality / image_* / audio_*")
print("  上下文：status / mode / history / agent_note / user_name")
print("  闸门产物：handoff_keyword / intent / emotion / perceive_reply / user_msg_persisted")
print("  检索产物：retrieved / top_score / screenshot_hint / context_text")
print("  生成产物：reply_raw / body / confidence / attachments / citations")
print("  护栏产物：need_handoff / handoff_reason / kb_nonempty")

  TurnState —— 输入 + 各节点产出（节点间靠它传递）
class TurnState(BaseModel):
    """Flow 的状态载体：输入 + 各节点产出，节点间通过 self.state 传递。"""

    # 输入
    conv_id: str
    user_id: str
    content: str = ""
    modality: str = "text"
    image_path: Optional[str] = None
    image_bytes: Optional[bytes] = None
    audio_bytes: Optional[bytes] = None
    # 用户上传图片/语音的 asset_id：用于落库 image_url/audio_url 并透传坐席（刷新不丢）
    image_asset: Optional[str] = None
    audio_asset: Optional[str] = None

    # preflight 注入的会话上下文
    status: str = "active"              # active / closed
    mode: str = "bot"                   # bot / waiting_human / human
    history: str = ""                   # 最近对话文本
    agent_note: str = ""                # 坐席备注
    user_name: str = ""
    asr_failed: bool = False            # 语音转写失败：提示重录，不落库/不检索

    # gate_intent 零 LLM 闸门产物
    handoff_keyword: bool = False

    # 意图感知（perceive_intent）。转人工的情绪判定只用 emotion 布尔
    # （落库 messages.emotion → get_negative_rounds），不需要 level/score 明细。
    intent: st

### 7.2 函数节点：零 LLM 的闸门

`gate_intent` 就是"三层意图分流"的前两层：**先判会话/模式，再用关键词短路转人工、寒暄预过滤**。
只有都不命中，才进入 `GATE_PERCEIVE` 交给模型。

In [24]:
# ============================================================
# Cell 24：函数节点 gate_intent / route_intent 源码
# ============================================================
print("=" * 66); print("  gate_intent（@router 函数）—— 模式 + 关键词闸门"); print("=" * 66)
print(inspect.getsource(session_flow.SessionTurnFlow.gate_intent))
print("=" * 66); print("  route_intent（@router 函数）—— 按 intent 分流"); print("=" * 66)
print(inspect.getsource(session_flow.SessionTurnFlow.route_intent))
print("要点：")
print("  - 关闭/人工中/语音失败/关键词转人工/寒暄，都在这里零 LLM 被分流出去")
print("  - 寒暄复用 ROUTE_CHITCHAT 事件名，让 handle_chitchat 单点监听即可命中")

  gate_intent（@router 函数）—— 模式 + 关键词闸门
    @router("preflight", emit=GATE_ROUTE_KEYS)
    async def gate_intent(self) -> str:
        """函数节点：先判会话/模式，再走三层意图分流的「零 LLM」前两层。"""
        from core.service.session_router import _is_chitchat
        st = self.state

        if st.status == "closed":
            return GATE_CLOSED
        if st.mode in ("waiting_human", "human"):
            return GATE_HUMAN
        if st.asr_failed:
            return GATE_ASR_FAILED

        st.handoff_keyword = await handoff_mod.detect_handoff_intent(st.content)
        if st.handoff_keyword:
            return GATE_HANDOFF_KW
        if _is_chitchat(st.content):
            st.intent = "chitchat"
            # 复用感知路径的寒暄事件名，让 handle_chitchat 单点监听即命中，
            # 避免多条寒暄事件需要 or_ 组合（or_ 在一次性 kickoff 竞态复杂）。
            return ROUTE_CHITCHAT
        return GATE_PERCEIVE

  route_intent（@router 函数）—— 按 intent 分流
    @router("perceive_intent", emit=ROUTE_KEYS)
    async def route_intent(self) -> str:
        ""

### 7.3 Agent 节点：单 Agent · 无工具（意图感知）

`perceive_intent` 是一个**真 Agent**，但**不给它任何工具**——没有工具就不会进入工具调用循环，
等价于一次结构化 LLM 往返。LLM 未配置或调用失败时，回退到 `perceive_message`（同一次往返）或保守 `it_issue`。

还有个细节：**纯截图消息（正文为空）直接判 `it_issue`**，跳过 LLM 感知。
否则空文本会被判成 `off_topic`，检索与截图分析都不会执行。

In [25]:
# ============================================================
# Cell 25：Agent 节点 perceive_intent 源码
# ============================================================
print(inspect.getsource(session_flow.SessionTurnFlow.perceive_intent))
print("要点：")
print("  - tools=[]：无工具 → 不进入工具循环，就是一次结构化 LLM 往返")
print("  - agent.execute_task(task) 是单 Agent 执行单个 Task 的正确入口（CrewAI 无 agent.run）")
print("  - 失败三级兜底：execute_task 异常 → perceive_message → PERCEIVE_FALLBACK")

    @listen(GATE_PERCEIVE)
    async def perceive_intent(self) -> None:
        """Agent 节点（无工具）：真 Agent 实例产出 intent + emotion + 短回复。

        教学演示「单 Agent · 无工具」：CrewAI Agent 无工具可调，不会进入工具
        调用循环，等价于一次结构化 LLM 往返。LLM 未配置或调用失败时回退到
        perceive_message（同一次 LLM 往返）或保守 intent=it_issue。
        """
        from core.rag.multimodal import (
            PERCEIVE_FALLBACK,
            _PERCEIVE_PROMPT,
            perceive_message,
        )
        st = self.state

        # 纯截图消息（正文为空）：截图本身即 IT 问题，跳过 LLM 感知直接判 it_issue。
        # 否则空文本会被感知模型判成 off_topic/chitchat，route_intent 随即走寒暄短回复，
        # 检索与截图分析都不会执行（回归：纯截图被判 off_topic 后只回了一句开场白）。
        if (st.image_bytes or st.image_path) and not (st.content or "").strip():
            st.intent = "it_issue"
            st.emotion = "neutral"
            st.perceive_reply = ""
            await _persist_user_message(st)
            _log_flow_step(self.recorder, "perceive_intent",
                           {"intent": "it_issue", "skip_llm"

### 7.4 Agent 节点：单 Agent · 单工具（知识库检索）

`prepare_context` 里截图分析是**前置函数计算**（有图就分析，没有决策空间，不占独立节点）；
真正的检索交给一个只配了 `MultimodalSearchTool` 的 Agent——由模型**自主决定检索词**。

In [26]:
# ============================================================
# Cell 26：Agent 节点 prepare_context 源码
# ============================================================
print(inspect.getsource(session_flow.SessionTurnFlow.prepare_context))
print("要点：")
print("  - 截图预分析：process_image 失败时改用'中性提示'，不把内部错误暴露给用户")
print("  - 检索 Agent 失败时，回退主线程直接调 multimodal_rag.search（兜底不中断）")
print("  - 检索结果以 recorder 为准（Agent 工具或兜底检索都会写回 recorder）")

    @listen(ROUTE_IT_ISSUE)
    async def prepare_context(self) -> None:
        """Agent 节点（单工具 multimodal_search）：自主检索知识库，装配上下文。

        截图分析无决策空间（有图就分析），作为前置函数计算，不占独立节点。
        检索由 multimodal_rag.search 完成（Agent 经 MultimodalSearchTool 调用），
        结果写入 state，供 generate_reply 使用并作护栏依据。
        """
        st = self.state

        # 截图预分析（前置函数计算，无决策空间）。
        # 截图链路：ws_api 只传 image_path（不传 image_bytes），
        # 这里统一用 image_bytes 优先、再以 _read_bytes(image_path) 兜底补读，
        # 保证截图分析不因字节未传入而失效（与老编排的 _read_bytes 一致）。
        screenshot_bytes = st.image_bytes or _read_bytes(st.image_path)
        if screenshot_bytes:
            try:
                shot = await process_image(screenshot_bytes)
                self.recorder.screenshot = shot  # type: ignore[union-attr]
                # process_image 视觉理解失败时不抛异常，而是在 description 里写
                # 「视觉理解失败：…」。原样注入会被模型转述给用户（回归：出现
                # 「系统在视觉理解时遇到了网络超时」并让用户重发截图）。失败时改用
                # 中性提示，引导其依据知识库作答且不暴露内部错误。
             

### 7.5 Crew 节点：多 Agent 协作产出回复

`generate_reply` 复用 `crew_agents.run_crew_turn`：
- `CREW_ENABLED=true` 走 CrewAI（一线合成 + 按需工单 + 质检），流式逐 token 推送；
- 否则（或 Crew 失败）走中台**直连 LLM 流式**——两条路的推送事件完全一致。

In [27]:
# ============================================================
# Cell 27：Crew 节点 generate_reply 源码
# ============================================================
print(inspect.getsource(session_flow.SessionTurnFlow.generate_reply))
print("要点：")
print("  - _on_delta 把每个 token 推成 bot_stream_delta，与旧编排同事件格式")
print("  - Crew 路径未检索到时，补一次兜底检索，仅供护栏使用（不作正文依据）")

    @listen("prepare_context")
    async def generate_reply(self) -> None:
        """Crew 节点：复用现有 crew_agents.run_crew_turn，多 Agent 协作产出回复。

        Crew 内部（见 crew_agents.py）：
            - 一线合成 —— 基于 state.retrieved 把知识翻译成排查步骤，
                         判断检索是否够用，不够令 [ESCALATE: yes]；
            - 工单处理 —— 按需触发，仅 [ESCALATE: yes] 时执行建单/改密/权限；
            - 质检     —— 默认关闭，旁路复核。
        流式：on_delta 逐 token 推 bot_stream_delta，与现有编排共用同一套事件。
        """
        st = self.state

        async def _on_delta(text: str) -> None:
            await ws_manager.send_to_user(
                st.user_id,
                {"type": "bot_stream_delta", "conversation_id": st.conv_id,
                 "data": {"delta": text}},
            )

        user_parts = [f"【知识库检索结果】\n{st.context_text}"]
        if st.agent_note:
            user_parts.append(f"【人工坐席备注】{st.agent_note}")
        if st.history:
            user_parts.append(f"【最近对话】\n{st.history}")
        if st.screenshot_hint:
            user_parts.

### 7.6 函数节点：置信度护栏 + 装配交付

`gate_and_deliver` 是本轮的"最后一公里"：
调 `should_handoff` 判护栏 → 装配 citations/attachments → 生成 TTS 语音 →
落库 → 推 `bot_stream_end` → 决定 `DELIVER_PASS` 还是 `DELIVER_HANDOFF`。

有一个诚实的工程约定：**`answer_grounded = (not CREW_ENABLED) or bool(retrieved)`**——
直连路径的回复不做"有无检索"的装配限制；Crew 路径若本轮没检索到，就不装配引用（避免无依据却标来源）。

In [28]:
# ============================================================
# Cell 28：护栏与交付 gate_and_deliver 源码
# ============================================================
print(inspect.getsource(session_flow.SessionTurnFlow.gate_and_deliver))
print("要点：")
print("  - should_handoff 的 turn_confidence 只在'本轮有检索'时传，否则传 None（交给规则 4）")
print("  - TTS 失败静默降级为纯文本，不阻断主流程")
print("  - 落库后才推送，保证前端刷新/切换会话后消息不丢")

    @router("generate_reply", emit=DELIVER_KEYS)
    async def gate_and_deliver(self) -> str:
        """函数节点：置信度护栏判定 + 装配引用/附件 + 入库 + 推送。"""
        st = self.state

        if not st.reply_raw:
            st.reply_raw = "抱歉，AI 服务暂时不可用。已为您记录问题，将转交人工处理。"

        answer_grounded = (not settings.CREW_ENABLED) or bool(st.retrieved)
        st.kb_nonempty = vector_store.count() > 0

        need, reason = await handoff_mod.should_handoff(
            st.conv_id,
            turn_confidence=st.top_score if st.retrieved else None,
            user_text=st.content,
            kb_nonempty=st.kb_nonempty,
            intent=st.intent,
        )
        st.need_handoff = need
        st.handoff_reason = reason

        if answer_grounded:
            st.attachments = multimodal_rag.build_attachments(st.retrieved) if st.retrieved else []
            st.citations = multimodal_rag.build_citations(st.retrieved) if st.retrieved else []
            st.body = _append_citations(st.reply_raw, st.retri

### 7.7 TurnRecorder：旁路记账（不进入 Agent 上下文）

工具在 Agent 执行期间把检索结果/截图/工单调用写进 `recorder`；
Agent 结束后，路由从 recorder 读出命中来源，装配 `citations / attachments`，
并用于低置信度护栏判定。

**关键点：recorder 是旁路的**——它不喂给模型，只服务于"装配与审计"。

In [29]:
# ============================================================
# Cell 29：TurnRecorder 源码 + 实测记账
# ============================================================
from core.service.turn_context import TurnRecorder

print(inspect.getsource(TurnRecorder))

rec = TurnRecorder(conversation_id="c-demo", user_id="u-1")
rec.record_tool_call("multimodal_search", {"query": "VPN"}, "命中 3 条", 420)
rec.record_tool_call("create_ticket", {"title": "VPN故障"}, "TICKET-abc12345", 88)
rec.retrieved = [{"id": "c1", "score": 0.42, "metadata": {"modality": "text"}}]
rec.top_score = 0.42

print()
print("实测：")
print(f"  conversation_id = {rec.conversation_id}")
print(f"  top_score       = {rec.top_score}")
print(f"  retrieved 条数   = {len(rec.retrieved)}")
print("  tool_calls：")
for call in rec.tool_calls:
    print(f"    {call['tool_name']:<20} {call['status']:<8} {call['latency_ms']}ms  → {call['output_result']}")

@dataclass
class TurnRecorder:
    """一轮对话的旁路记录（不进入 Agent 上下文）。"""

    conversation_id: str
    user_id: str = ""
    # 最近一次多模态检索结果（[{id, score, metadata}]）
    retrieved: list[dict] = field(default_factory=list)
    # 最近一次检索的 top1 分数（置信度护栏用，见 6.3）
    top_score: float = 0.0
    # 截图分析结果（process_image 的返回值）
    screenshot: Optional[dict] = None
    # 工单相关
    tickets: list[str] = field(default_factory=list)
    tool_calls: list[dict] = field(default_factory=list)

    def record_tool_call(self, tool_name: str, params: Any, result: Any, latency_ms: int, status: str = "success") -> None:
        self.tool_calls.append(
            {
                "tool_name": tool_name,
                "input_params": params,
                "output_result": result,
                "latency_ms": latency_ms,
                "status": status,
            }
        )


实测：
  conversation_id = c-demo
  top_score       = 0.42
  retrieved 条数   = 1
  tool_calls：
    multimodal_search    success  420ms  → 命中 

---
## Part 8：人机协同 —— 会话生命周期与坐席动作
---

### 8.1 三个正交维度：别把它们混成一个状态

会话的"状态"其实是**三个互相独立的维度**，混在一起写会导致状态爆炸：

| 维度 | 取值 | 含义 |
|------|------|------|
| `conversations.status` | `active` / `closed` | 会话是否还活着 |
| `conversations.mode` | `bot` / `waiting_human` / `human` | 当前由谁服务 |
| `tickets.status` | `open` / `in_progress` / `resolved` / `closed` | 工单进度 |

> 关键约定：**会话关闭**只由"用户显式结束"或"30 分钟超时"触发；
> 坐席"结束服务"只是把 `mode` 切回 `bot`，**不关闭会话**。工单与会话状态**不级联**。

### 8.2 零 LLM 预过滤：`_is_chitchat`

在进入感知之前，先用纯字符串规则做一次寒暄预过滤——命中就连感知那次 LLM 调用都省掉。
实现很讲究：**剔除寒暄词与标点后看剩余内容**，只有剩余不超过 2 个字符才算寒暄。
所以"你好，我的VPN连不上"剩余"我的VPN连不上"，**不是寒暄**，正常走 RAG。

In [30]:
# ============================================================
# Cell 30：_is_chitchat 源码 + 实测
# ============================================================
from core.service.session_router import _is_chitchat

print(inspect.getsource(_is_chitchat))
print()
print("实测：")
for text in ["你好", "您好啊", "谢谢", "hi there", "ok",
             "你好，我的VPN连不上", "麻烦帮我转人工"]:
    print(f"  {text!r:<22} → 寒暄={_is_chitchat(text)}")

def _is_chitchat(text: str) -> bool:
    """判断是否寒暄/口水话：剔除寒暄词与标点后没有实质内容。

    「你好，我的VPN连不上」剩余「我的VPN连不上」→ 不是寒暄，正常走 RAG。
    """
    raw = (text or "").strip()
    if not raw:
        return False
    stripped = raw.strip(_CHITCHAT_STRIP)
    if stripped.isascii() and stripped:
        tokens = stripped.lower().replace(",", " ").replace(".", " ").split()
        if tokens and all(token in _CHITCHAT_LATIN for token in tokens):
            return True
    residue = raw
    for keyword in CHITCHAT_KEYWORDS:
        residue = residue.replace(keyword, "")
    return len(residue.strip(_CHITCHAT_STRIP)) <= _CHITCHAT_MAX_RESIDUE


实测：
  '你好'                   → 寒暄=True
  '您好啊'                  → 寒暄=True
  '谢谢'                   → 寒暄=True
  'hi there'             → 寒暄=True
  'ok'                   → 寒暄=True
  '你好，我的VPN连不上'          → 寒暄=False
  '麻烦帮我转人工'              → 寒暄=False


### 8.3 `trigger_handoff`：有坐席 / 无坐席，两条分支

转人工的入口只有一个，但内部分两条路。顺序经过精心设计——**先给用户可见反馈，再做耗时操作**：

```
查可用坐席
 ├─ 无坐席：切 waiting_human → 提示"正在转接" → 建单（含一次分类 LLM）→ 回 bot → 回执工单号
 └─ 有坐席：切 waiting_human → 提示"正在转接" → 生成交接摘要（含 LLM）→ 推给坐席
```

> 为什么"无坐席兜底"分支要**排在摘要生成之前**？因为兜底路径**不消费摘要**，
> 没必要白跑一次 LLM（省成本与延迟）。

In [31]:
# ============================================================
# Cell 31：trigger_handoff 源码
# ============================================================
import core.service.session_router as session_router

print(inspect.getsource(session_router.trigger_handoff))
print("要点：")
print("  - 可用坐席判定用 db.list_available_agents()（status=online 且未满员），不只看 WS 连接")
print("  - 无坐席分支：先推 handoff_triggered 让用户看到反馈，再建单，最后回 bot + 回执")
print("  - 有坐席分支：先切 waiting_human + 通知用户，再生成摘要推坐席（用户不必干等 LLM）")
print("  - 建单前先 list_conversation_tickets 去重，避免多次触发重复建单")

async def trigger_handoff(conv_id: str, reason: str) -> dict:
    conversation = await db.get_conversation(conv_id)
    if not conversation or conversation["status"] == "closed":
        return {"ok": False}

    # 无可用坐席兜底：提示繁忙 + 自动建单 + 回到 bot（见 6.8）。
    # 判定口径与坐席状态一致（status='online' 且未满员）：只看 WS 连接数会把
    # 已切到 busy/offline 的坐席也算作「有人」，导致漏建单。
    # 分支先于摘要生成：兜底路径不消费摘要，不必为此白跑一次 LLM（省成本与延迟）
    available_agents = await db.list_available_agents()
    if not available_agents:
        # 先给用户即时反馈（切 waiting_human + 提示「正在转接」），再建单。
        # 建单含一次分类 LLM 调用，排在用户可见反馈之后，用户不必干等
        await db.update_conversation(conv_id, mode="waiting_human", handoff_reason=reason)
        await _push_user(
            conv_id, conversation["user_id"],
            {"type": "handoff_triggered", "conversation_id": conv_id,
             "data": {"role": "system", "content": "正在为您转接人工客服，请稍候…",
                      "mode": "waiting_human", "reason": reason}},
        )
        # 去重：同一会话已有工单则复用，避免多次触发转人工时重复建单（口径同 close

### 8.4 坐席动作：接入 / 回复 / 交还 AI / 结束服务

坐席的工作流是"接入 → 回复 → 交还 AI"，每个动作都有**归属校验**，
避免越权向他人会话注入消息。

一个关键约定：**接入/交还都不改写坐席自己的状态**（在线/忙碌/离线由坐席显式切换），
只调用 `sync_agent_current_count()` **按事实重算**接待数。

In [32]:
# ============================================================
# Cell 32：坐席动作源码
# ============================================================
print("=" * 66); print("  agent_join —— 接入会话"); print("=" * 66)
print(inspect.getsource(session_router.agent_join))
print("=" * 66); print("  return_to_ai —— 交还 AI（可带备注）"); print("=" * 66)
print(inspect.getsource(session_router.return_to_ai))
print("=" * 66); print("  end_agent_service —— 坐席'结束会话'（只切 mode=bot，不关闭会话）"); print("=" * 66)
print(inspect.getsource(session_router.end_agent_service))
print("要点：")
print("  - return_to_ai 里 note 会落成 [人工坐席备注] 系统消息，供 AI 下一轮读取上下文")
print("  - 三者都只 sync_agent_current_count 重算接待数，不改写坐席状态")

  agent_join —— 接入会话
async def agent_join(conv_id: str, agent_id: str) -> dict:
    conversation = await db.get_conversation(conv_id)
    if not conversation:
        return {"ok": False, "error": "会话不存在"}
    if conversation["mode"] != "waiting_human":
        return {"ok": False, "error": "该会话当前不在等待人工状态"}

    await db.update_conversation(conv_id, mode="human", assigned_agent_id=agent_id)
    # 状态由坐席自己切换，接入不改写；只按事实重算接待数（见 6.8）
    await db.sync_agent_current_count(agent_id)
    await db.add_message(conv_id, "system", "人工客服已接入")
    await _push_user(
        conv_id, conversation["user_id"],
        {"type": "agent_joined", "conversation_id": conv_id,
         "data": {"role": "system", "content": "人工客服已接入", "mode": "human"}},
    )
    await _push_agents(
        {"type": "queue_update", "conversation_id": conv_id,
         "data": {"joined": conv_id, "agent_id": agent_id}}
    )
    await _push_admins_stats()
    return {"ok": True, "mode": "human"}

  return_to_ai —— 交还 AI（可带备注）
as

### 8.5 会话关闭与建单口径

`close_conversation` 处理两种关闭原因：`resolved`（问题解决）与 `user_closed`（用户主动结束）。
其中 `user_closed` 有一条重要规则：**若问题未解决且无关联工单，自动建单并回执工单号**——
避免"用户关了会话，问题就丢了"。

In [33]:
# ============================================================
# Cell 33：close_conversation 源码
# ============================================================
print(inspect.getsource(session_router.close_conversation))
print("要点：")
print("  - user_closed 且无关联工单 → 自动建单（extra 注明'用户主动结束且未解决'）")
print("  - 已有工单则复用（list_conversation_tickets 去重），不重复建单")
print("  - 关闭时若坐席名下还有该会话，sync_agent_current_count 回退接待数")

async def close_conversation(conv_id: str, reason: str) -> dict:
    """关闭会话（见 6.9）。reason: resolved / user_closed。

    user_closed 且无关联工单时自动建单并回执工单号。
    """
    conversation = await db.get_conversation(conv_id)
    if not conversation:
        return {"ok": False, "error": "会话不存在"}
    if conversation["status"] == "closed":
        return {"ok": True, "already_closed": True}

    ticket_id: Optional[str] = None
    if reason == "user_closed":
        existing = await handoff_mod.list_conversation_tickets(conv_id)
        if existing:
            ticket_id = existing[-1]
        else:
            ticket_id = await handoff_mod.create_ticket_for_conversation(
                conv_id, conversation["user_id"], extra="用户主动结束会话且问题未解决"
            )

    await db.update_conversation(
        conv_id, status="closed", end_reason=reason, ended_at=now_iso(), mode="bot"
    )
    # 用户可能在人工服务中直接关闭会话，坐席接待数需同步回退（见 6.8）
    if conversation.get("assigned_agent_id"):
        await db.sync_agent_curre

---
## Part 9：MCP 工单系统 —— 唯一的外部系统
---

### 9.1 为什么工单要走 MCP，而检索不用？

- **检索**是本项目自己的进程内能力 → 进程内 Tool 即可（Part 4 已讲）；
- **工单**是"别人的系统"（独立进程 + 独立数据库）→ 用 **MCP（Model Context Protocol）** 解耦。

```
中台(MCP Host)  ──stdio──►  工单 MCP Server（独立子进程，FastMCP + SQLite）
   mcp_client / mcp_bridge            mcp_servers/it_service/server.py
```

MCP 的好处：Server 与 Host 之间只靠**协议 + 工具 schema** 通信。
Server 换语言、换实现，中台代码不用改；工具增删，中台**动态注册**（Part 4 的 `_args_model_from_schema`）。

### 9.2 三个必须解决的难题

直接把 MCP SDK 用在 FastAPI 里会踩三个坑：

| 难题 | 现象 | 本项目的解法 |
|------|------|--------------|
| **① anyio 同任务约束** | `stdio_client` 基于 anyio 任务组，必须**同一任务内进入/退出**；FastAPI 每个请求是不同任务，会触发 cancel scope 跨任务错误 | 用**一条常驻后台任务** `_run` 独占连接 |
| **② 跨线程调用** | CrewAI 的同步 Tool 跑在 worker 线程，不能直接 `await` 协程 | `call_tool_sync` 用 `run_coroutine_threadsafe` 投回主循环 |
| **③ 常驻生命周期** | 连接建/断要在同一任务，且要能优雅停机 | 队列投递 + `_STOP` 哨兵停机 |

### 9.3 常驻后台任务 + 队列桥

`MCPBridge.start()` 只创建队列和后台任务；真正的连接、调用、断开**都发生在 `_run` 这一个任务里**。
调用方把 `(操作, 参数, future)` 放进队列，`_run` 逐条取出执行，再把结果写回 future。

In [34]:
# ============================================================
# Cell 34：MCPBridge 生命周期源码
# ============================================================
import core.infra.mcp_bridge as mcp_bridge

print(mcp_bridge.__doc__)
print("=" * 66); print("  start —— 记录事件循环、建队列、拉起后台任务"); print("=" * 66)
print(inspect.getsource(mcp_bridge.MCPBridge.start))
print("=" * 66); print("  _run —— 唯一与连接打交道的任务（连接/调用/断开都在这里）"); print("=" * 66)
print(inspect.getsource(mcp_bridge.MCPBridge._run))
print("要点：")
print("  - 连接失败不退出进程：记录错误 + 置位 _ready，available 返回 False（优雅降级）")
print("  - 停机用 _STOP 哨兵（独一无二的对象，避免与正常载荷混淆）")
print("  - finally 里无论怎么退出都断开连接，避免留下僵尸子进程")

MCP 常驻连接桥。

问题：MCP 官方 SDK 的 stdio_client 基于 anyio 任务组，必须在**同一个 asyncio 任务**
内进入与退出。FastAPI 每个请求是不同任务，若在请求里 connect、在另一请求里
disconnect，会触发 cancel scope 跨任务错误。

方案：用一条常驻后台任务持有连接，所有工具调用通过队列投递给它执行。
CrewAI 的同步工具运行在线程中，可用 call_tool_sync() 通过
asyncio.run_coroutine_threadsafe 投递到主事件循环。

【这个文件在系统里的角色】
"工单 MCP Server"是一个独立进程，通过标准输入输出（stdio）与本进程通信、对外提供
工单查询/创建等工具。本桥负责"长期维持这一条连接"，并把上层（含 CrewAI 工具、路由）
的调用请求安全地转发过去、把结果带回来，是业务代码与 MCP 之间的唯一通道。

【关键概念扫盲】
1) 常驻后台任务：start() 只创建一条后台协程 _run，它独占持有 MCP 连接；
   其他协程想调用工具，不是自己连，而是把请求"投递"给它执行。
2) 队列通信：调用方把 (操作, 参数, future) 放进 asyncio.Queue，_run 逐条取出执行，
   再把结果写回 future；调用方 await 这个 future 即可拿到结果。这样所有对连接的
   读写都发生在 _run 这一个任务里，规避了 anyio 的跨任务限制。
3) 线程安全桥：CrewAI 的同步工具运行在工作线程（非事件循环），工作线程里没法直接
   await 协程，于是用 call_tool_sync 通过 run_coroutine_threadsafe 把协程"投"回
   主事件循环执行，再同步等待结果。

  start —— 记录事件循环、建队列、拉起后台任务
    async def start(self) -> None:
        """启动桥：记录事件循环、建队列、拉起后台任务。

        说明：
            get_running_loop() 拿到当前的事件循环并保存，供后续线程安全的同步调用使用。
            create_t

### 9.4 两个调用入口：异步 `call_tool` 与线程安全 `call_tool_sync`

- `call_tool`（async）：中台 async 代码用，创建 future 投递后 `await`；
- `call_tool_sync`（同步）：给 CrewAI 的**同步** Tool 在 worker 线程里用，
  通过 `run_coroutine_threadsafe` 把协程投回主事件循环，再 `future.result()` 同步等待。

配套的 `core/infra/async_bridge.py` 里的 `run_sync()` 做的也是同一件事（把协程投回主循环），
供检索等中台 async 能力被同步 Tool 调用。

In [35]:
# ============================================================
# Cell 35：call_tool / call_tool_sync / run_sync 源码
# ============================================================
print("=" * 66); print("  MCPBridge.call_tool —— 异步投递 + await future"); print("=" * 66)
print(inspect.getsource(mcp_bridge.MCPBridge.call_tool))
print("=" * 66); print("  MCPBridge.call_tool_sync —— 跨线程同步等待"); print("=" * 66)
print(inspect.getsource(mcp_bridge.MCPBridge.call_tool_sync))
print("=" * 66); print("  async_bridge.run_sync —— 同步世界中执行协程"); print("=" * 66)
import core.infra.async_bridge as async_bridge
print(inspect.getsource(async_bridge.run_sync))
print("要点：")
print("  - call_tool_sync 多给 5 秒缓冲，避免调度延迟先于协程自身 timeout 触发")
print("  - run_sync 优先投主循环；未注册主循环时才退回 asyncio.run（避免事件循环爆炸）")

  MCPBridge.call_tool —— 异步投递 + await future
    async def call_tool(self, name: str, arguments: dict[str, Any], timeout: float = 30.0) -> str:
        """异步调用某个 MCP 工具并返回其结果。

        参数：
            name: 工具名。
            arguments: 工具入参 dict。
            timeout: 等待结果的超时秒数。
        返回：
            工具执行结果（字符串）。
        异常：
            桥不可用/未启动/调用超时/工具报错时抛 MCPClientError 或 TimeoutError。
        说明：
            本方法只负责"投递"：创建一个 future，把 ("call", (name, arguments), future)
            放进队列后 await 该 future；真正的执行在后台任务 _run 里。
        """
        if self._error:
            raise MCPClientError(f"工单 MCP Server 不可用：{self._error}")
        if self._queue is None or self._loop is None:
            raise MCPClientError("MCP 桥尚未启动。")
        # future 是"将来会有结果的占位符"：后台任务执行完会把结果 set_result 到它上面。
        fut: asyncio.Future = self._loop.create_future()
        await self._queue.put(("call", (name, arguments), fut))
        return await asyncio.wait_for(fut, timeout=timeout)

  MCPBridge.call_tool_sy

### 9.5 MCP Server：8 个工单工具

工单 Server 用 FastMCP 注册 8 个工具，数据落 SQLite（`tickets.db`，可通过 `TICKETS_DB_PATH` 重定向）。
Server 侧**不另建用户表**，`user_id` 直接用中台 `users.id`。

| 工具 | 作用 |
|------|------|
| `create_ticket` | 创建工单（priority/category 非法值自动回退） |
| `query_ticket` | 按工单号查详情 |
| `list_user_tickets` | 查某用户全部工单 |
| `list_tickets` | 管理端分页查询全部工单 |
| `delete_ticket` | 删除工单 |
| `update_ticket_status` | 更新状态（校验枚举） |
| `reset_password` | 重置密码，返回一次性临时密码 |
| `request_software_access` | 提交软件权限申请 |

In [36]:
# ============================================================
# Cell 36：从源码提取 @mcp.tool() 工具清单 + create_ticket 源码
# ============================================================
import importlib.util

SERVER_PATH = "mcp_servers/it_service/server.py"

# 用 ast 找出所有被 @mcp.tool() 装饰的函数（不启动 Server）
tree = ast.parse(Path(SERVER_PATH).read_text(encoding="utf-8"))
tool_names = []
for node in tree.body:
    if isinstance(node, ast.FunctionDef):
        for dec in node.decorator_list:
            if isinstance(dec, ast.Call) and getattr(dec.func, "attr", "") == "tool":
                tool_names.append((node.name, node.lineno))

print("=" * 66)
print(f"  工单 MCP Server 暴露的工具（共 {len(tool_names)} 个）")
print("=" * 66)
for name, line in tool_names:
    print(f"  - {name:<24} (第 {line} 行)")

# 以独立模块方式加载 Server（不执行 mcp.run()）
spec = importlib.util.spec_from_file_location("it_server", SERVER_PATH)
it_server = importlib.util.module_from_spec(spec)
spec.loader.exec_module(it_server)

print()
print("=" * 66)
print("  create_ticket —— 建单（注意非法枚举的回退说明）")
print("=" * 66)
print(inspect.getsource(it_server.create_ticket))

  工单 MCP Server 暴露的工具（共 8 个）
  - create_ticket            (第 102 行)
  - query_ticket             (第 153 行)
  - list_user_tickets        (第 182 行)
  - list_tickets             (第 205 行)
  - delete_ticket            (第 265 行)
  - update_ticket_status     (第 291 行)
  - reset_password           (第 339 行)
  - request_software_access  (第 362 行)

  create_ticket —— 建单（注意非法枚举的回退说明）
@mcp.tool()
def create_ticket(
    title: str,
    description: str,
    priority: str = "medium",
    category: str = "general",
    user_id: str = "",
) -> str:
    """创建一个 IT 服务工单。

    Args:
        title: 工单标题（必填）。
        description: 问题描述。
        priority: 优先级，枚举 high/medium/low，默认 medium；非法值回退 medium。
        category: 分类，枚举 network/password/software/finance/email/general，
            默认 general；非法值回退 general。
        user_id: 发起工单的用户 id（中台 users.id，见 5.6），可为空。

    Returns:
        形如 "工单已创建：TICKET-xxxxxxxx" 的文本；若入参枚举非法，
        会在结果末尾附带回退说明。
    """
    notes: list[str] = []

    if priority not in PRIORI

### 9.6 实测：建单 → 查单 → 列单（用临时库，不污染真实工单库）

FastMCP 的 `@mcp.tool()` 装饰器会把原函数返回，所以我们可以**直接调用**这些工具函数来验证逻辑，
只需先把 `TICKETS_DB_PATH` 指到一个临时文件。

In [37]:
# ============================================================
# Cell 37：MCP 工具实测（临时库，离线）
# ============================================================
import os
import json

os.environ["TICKETS_DB_PATH"] = os.path.join(tempfile.mkdtemp(), "nb37_tickets.db")
it_server.ensure_db()

print("=" * 66)
print("  ① create_ticket（故意传非法优先级 urgent 与非法分类 printer）")
print("=" * 66)
r1 = it_server.create_ticket(
    title="VPN 无法连接", description="用户反馈 VPN 连不上，已尝试重启",
    priority="urgent", category="printer", user_id="u-1",
)
print("  ->", r1)
ticket_id = r1.split("TICKET-")[1][:8]
ticket_id = "TICKET-" + ticket_id

print()
print("=" * 66)
print("  ② query_ticket")
print("=" * 66)
print("  ->", it_server.query_ticket(ticket_id))

print()
print("=" * 66)
print("  ③ update_ticket_status（非法状态应被拒绝）")
print("=" * 66)
print("  非法状态 ->", it_server.update_ticket_status(ticket_id, "done"))
print("  合法状态 ->", it_server.update_ticket_status(ticket_id, "in_progress"))

print()
print("=" * 66)
print("  ④ list_user_tickets")
print("=" * 66)
print("  ->", it_server.list_user_tickets("u-1"))

print()
print("=" * 66)
print("  ⑤ 一次性密码 / 软件权限申请")
print("=" * 66)
print("  reset_password          ->", it_server.reset_password("u-1"))
print("  request_software_access ->", it_server.request_software_access("u-1", "Visual Studio Code"))

  ① create_ticket（故意传非法优先级 urgent 与非法分类 printer）
  -> 工单已创建：TICKET-d3239d3d（优先级 'urgent' 非法，已回退为 medium；分类 'printer' 非法，已回退为 general）

  ② query_ticket
  -> {"id": "TICKET-d3239d3d", "user_id": "u-1", "title": "VPN 无法连接", "description": "用户反馈 VPN 连不上，已尝试重启", "priority": "medium", "category": "general", "status": "open", "created_at": "2026-10-04 15:27:24", "updated_at": "2026-10-04 15:27:24", "found": true}

  ③ update_ticket_status（非法状态应被拒绝）
  非法状态 -> {"ok": false, "message": "非法状态：done，可选值：['closed', 'in_progress', 'open', 'resolved']"}
  合法状态 -> {"ok": true, "ticket": {"id": "TICKET-d3239d3d", "user_id": "u-1", "title": "VPN 无法连接", "description": "用户反馈 VPN 连不上，已尝试重启", "priority": "medium", "category": "general", "status": "in_progress", "created_at": "2026-10-04 15:27:24", "updated_at": "2026-10-04 15:27:24"}}

  ④ list_user_tickets
  -> [{"id": "TICKET-d3239d3d", "user_id": "u-1", "title": "VPN 无法连接", "description": "用户反馈 VPN 连不上，已尝试重启", "priority": "medium", "category": "general",

### 9.7 小结：外部系统接入的三层

| 层 | 文件 | 职责 |
|----|------|------|
| 客户端 | `mcp_client.py` | 拉起子进程、握手、发现工具、标准化调用，异常统一包成 `MCPClientError` |
| 桥 | `mcp_bridge.py` | 常驻连接 + 队列投递 + 线程安全同步调用（解决 anyio 同任务约束） |
| 服务端 | `mcp_servers/it_service/server.py` | FastMCP 注册 8 个工具，落 SQLite |

**"唯一外部系统"的纪律**：除工单外，感知/RAG/检索全部是进程内能力，
所以整个系统只有**一条**外部协议边界——这让系统更好理解、更好部署。

---
## Part 10：实战任务 + 随堂练习（含答案）
---

### 10.1 实战任务

| 任务 | 做什么 | 观察点 |
|------|--------|--------|
| **对比双路径** | 配置 `CREW_ENABLED=true` 发一条消息；再改 `false` 发同样的消息 | 两者前端收到的事件一致；日志里三角色任务分解是否出现 |
| **走通转人工** | 用户说"转人工" | 关键词短路（<2s 收到 handoff_triggered）；有坐席→摘要推送；无坐席→建单回执 |
| **验证工单工具** | 用 MCP Inspector 或工具日志 | 8 个工具的调用与审计落库（`tool_calls`） |
| **跑回归测试** | `pytest -q tests/test_handoff.py` | 四类触发规则 + 超时关闭全绿 |

### 10.2 常见误区与答疑

| 误区 | 纠正 |
|------|------|
| "转人工就是加个 `if '转人工' in text`" | 是**四条按序规则**，关键词只是第 1 条（零 LLM 短路） |
| "低置信度就该转人工" | 三个前置条件缺一不可：有检索 + `it_issue` + `0<分数<阈值`；空库/寒暄必须排除 |
| "连续几轮没解决"直接数消息数 | 只数**IT 问题**轮次，寒暄/无关话题会打断计数 |
| "转人工要先跑摘要" | 先切状态 + 通知用户，**再**生成摘要——用户不必干等 LLM |
| "坐席结束服务 = 关闭会话" | 只切 `mode=bot`，会话仍 `active`；关闭只由用户显式结束或超时触发 |
| "CrewAI 不兼容就换框架" | 先判断问题在**框架层还是 provider 层**；本项目两个坑都在 provider 类上补丁解决 |

### 10.3 随堂练习一：为什么关键词表里没有裸词「人工」？

请运行代码对比 `detect_handoff_intent("请帮我转人工")` 与 `detect_handoff_intent("人工智能是什么")`，
并解释为什么 `HANDOFF_KEYWORDS` 刻意不放裸词「人工」。

In [38]:
# ============================================================
# Cell 38：练习一参考思路（含答案）
# ============================================================
import asyncio

print("实测：")
for text in ["请帮我转人工", "人工智能是什么", "我要找真人"]:
    print(f"  {text!r:<16} → {asyncio.run(handoff_mod.detect_handoff_intent(text))}")

answer = """
原因：裸词「人工」会命中大量无关词——「人工智能」「人工湖」「人工降雨」等。

所以关键词表只放'搭配了动词/称谓'的组合，例如：
  转人工 / 接人工 / 人工客服 / 人工服务 / 找人工 / 要人工 / 换人工 / 人工介入 ...
以及同义表达：真人 / 找客服 / 转客服 / 找工程师 / 找人处理 ...

结论：
  - '人工智能是什么' 里虽然含「人工」，但不是转人工诉求 → 不能命中；
  - 关键词匹配是'零 LLM 快速短路'，宁可少配几个词，也不能让误命中把正常咨询转给人工。
  - 关键词无法穷举的部分，交给 perceive_message 用大模型判 intent 兜底。
"""
print(answer)

实测：
  '请帮我转人工'         → True
  '人工智能是什么'        → False
  '我要找真人'          → True

原因：裸词「人工」会命中大量无关词——「人工智能」「人工湖」「人工降雨」等。

所以关键词表只放'搭配了动词/称谓'的组合，例如：
  转人工 / 接人工 / 人工客服 / 人工服务 / 找人工 / 要人工 / 换人工 / 人工介入 ...
以及同义表达：真人 / 找客服 / 转客服 / 找工程师 / 找人处理 ...

结论：
  - '人工智能是什么' 里虽然含「人工」，但不是转人工诉求 → 不能命中；
  - 关键词匹配是'零 LLM 快速短路'，宁可少配几个词，也不能让误命中把正常咨询转给人工。
  - 关键词无法穷举的部分，交给 perceive_message 用大模型判 intent 兜底。



### 10.4 随堂练习二：给转人工判定加一条新规则

产品说："最近 2 轮都给用户回了'抱歉没找到'，也应该转人工。"

请回答：
1. `should_handoff` 里已经有"连续未解决"（规则 4，≥3 轮）。你还需要新加规则吗？
2. 如果要新增，它应该插在**第几条**、依赖哪些数据？
3. 为什么**不能**把它放在规则 3 之前？

In [39]:
# ============================================================
# Cell 39：练习二参考思路（含答案）
# ============================================================
answer = """
1. 本项目的规则 4 用 HANDOFF_UNRESOLVED_ROUNDS 控制阈值（`.env` 可配，默认 3）。
   '连续 2 轮未解决'只是把该阈值改成 2，**不需要新增规则**——
   这正是把阈值抽成配置项的价值：改行为不改代码。
   改法：.env 里设 HANDOFF_UNRESOLVED_ROUNDS=2，然后**重启服务**（改 .env 不触发 --reload）。

2. 如果确实要一条语义不同的新规则（比如'同一会话连续两次低置信度'），
   正确做法是复用 get_unresolved_rounds / 历史 confidence 的统计口径，
   放在**规则 4 的位置**（连续类规则靠后），依赖 messages.confidence 与 intent。

3. 不能放在规则 3 之前的原因：规则顺序 = '从最便宜/最确定到最昂贵'。
   - 规则 1 零 LLM；规则 2/4 各一次 SQL；规则 3 依赖本轮检索结果。
   - 连续类规则要扫历史，成本高于'本轮单次判断'；
   - 更重要的是：低置信度规则有'本轮有检索'这个前置条件，
     若把连续规则提前，会让'本轮其实答得好、但历史有几轮未解决'的用户被过早转人工。
"""
print(answer)


1. 本项目的规则 4 用 HANDOFF_UNRESOLVED_ROUNDS 控制阈值（`.env` 可配，默认 3）。
   '连续 2 轮未解决'只是把该阈值改成 2，**不需要新增规则**——
   这正是把阈值抽成配置项的价值：改行为不改代码。
   改法：.env 里设 HANDOFF_UNRESOLVED_ROUNDS=2，然后**重启服务**（改 .env 不触发 --reload）。

2. 如果确实要一条语义不同的新规则（比如'同一会话连续两次低置信度'），
   正确做法是复用 get_unresolved_rounds / 历史 confidence 的统计口径，
   放在**规则 4 的位置**（连续类规则靠后），依赖 messages.confidence 与 intent。

3. 不能放在规则 3 之前的原因：规则顺序 = '从最便宜/最确定到最昂贵'。
   - 规则 1 零 LLM；规则 2/4 各一次 SQL；规则 3 依赖本轮检索结果。
   - 连续类规则要扫历史，成本高于'本轮单次判断'；
   - 更重要的是：低置信度规则有'本轮有检索'这个前置条件，
     若把连续规则提前，会让'本轮其实答得好、但历史有几轮未解决'的用户被过早转人工。



### 10.5 随堂练习三：给工单 Server 增加一个工具

产品说："管理端要能看**超时未处理**的工单。"

请设计：如何在工单 MCP Server 增加一个 `list_overdue_tickets` 工具？中台需要改代码吗？

In [40]:
# ============================================================
# Cell 40：练习三参考思路（含答案）
# ============================================================
answer = """
一、Server 侧（mcp_servers/it_service/server.py）：新增一个带 @mcp.tool() 的函数

    @mcp.tool()
    def list_overdue_tickets(hours: int = 24) -> str:
        "查询创建超过 hours 小时仍未 closed 的工单。"
        conn = _connect()
        try:
            rows = conn.execute(
                "SELECT * FROM tickets WHERE status != 'closed' "
                "AND created_at <= datetime('now', ?) "
                "ORDER BY created_at ASC",
                (f'-{int(hours)} hours',),
            ).fetchall()
        finally:
            conn.close()
        return json.dumps([dict(r) for r in rows], ensure_ascii=False)

注意：工具函数的 docstring 会作为工具描述暴露给模型；参数类型注解会被写进 inputSchema。

二、中台侧：**基本不用改代码**。
   - MCPBridge 启动时会 list_tool_schemas() 发现新工具并缓存；
   - build_mcp_tools() 按 schema 动态生成 pydantic 入参模型（见 Part 4）；
   - 只要把 'list_overdue_tickets' 加进某个角色工厂的 names 白名单（如管理端/工单 Agent），
     它就会被注册成可调用工具。

结论：这就是 MCP + 动态注册的价值——**外部系统加能力，中台几乎零改动**，
唯一要动的是决定'哪个角色可以调用它'的白名单。
"""
print(answer)


一、Server 侧（mcp_servers/it_service/server.py）：新增一个带 @mcp.tool() 的函数

    @mcp.tool()
    def list_overdue_tickets(hours: int = 24) -> str:
        "查询创建超过 hours 小时仍未 closed 的工单。"
        conn = _connect()
        try:
            rows = conn.execute(
                "SELECT * FROM tickets WHERE status != 'closed' "
                "AND created_at <= datetime('now', ?) "
                "ORDER BY created_at ASC",
                (f'-{int(hours)} hours',),
            ).fetchall()
        finally:
            conn.close()
        return json.dumps([dict(r) for r in rows], ensure_ascii=False)

注意：工具函数的 docstring 会作为工具描述暴露给模型；参数类型注解会被写进 inputSchema。

二、中台侧：**基本不用改代码**。
   - MCPBridge 启动时会 list_tool_schemas() 发现新工具并缓存；
   - build_mcp_tools() 按 schema 动态生成 pydantic 入参模型（见 Part 4）；
   - 只要把 'list_overdue_tickets' 加进某个角色工厂的 names 白名单（如管理端/工单 Agent），
     它就会被注册成可调用工具。

结论：这就是 MCP + 动态注册的价值——**外部系统加能力，中台几乎零改动**，
唯一要动的是决定'哪个角色可以调用它'的白名单。



### 10.6 本节小结

| 环节 | 文件 | 核心设计 |
|------|------|----------|
| 多 Agent | `crew_agents.py` | 三角色 + 顺序流程 + `[ESCALATE]` 按需工单 + 质检旁路 |
| 供应商兼容 | `crew_agents.py` | 两个 provider 类级补丁（ReAct 回退 / 剥流式 usage） |
| 流式 | `crew_agents.py` | 只推一线 TEXT 块 + 尾部扣留剥离标记 |
| 工具 | `crew_tools.py` | 进程内 Tool + 由 schema 动态生成 MCP Tool；保留最优命中 |
| 感知 | `multimodal.perceive_message` | 一次 LLM 出 intent+emotion+短回复；失败兜底 it_issue |
| 护栏 | `handoff.py` | 四条按序规则；关键词零 LLM 短路；连续计数按 intent 过滤 |
| 建单 | `handoff.py` | LLM 分类 + 审计落库 + 靠 tool_calls 去重 |
| 编排 | `session_flow.py` | Flow 节点 DAG；函数/Agent/Crew 三类节点；失败双重降级 |
| 记账 | `turn_context.py` | TurnRecorder 旁路记录，供装配与审计 |
| 协同 | `session_router.py` | status/mode/tickets 三维正交；坐席归属校验；接待数按事实重算 |
| 工单 | `mcp_client/bridge` + `server.py` | 常驻连接 + 队列桥 + 线程安全调用；8 个工具 |

### 10.7 交付物 / 验收

- CrewAI 三角色编排可用（`CREW_ENABLED=true` 时 Agent 自主调用检索/截图/工单工具）；
- 转人工闭环：关键词短路 → 有坐席通知 + 摘要 / 无坐席建单 + 回执；
- MCP 工单 Server 8 个工具可被调用，调用与审计落库；
- `tests/test_handoff.py` 全绿（四类触发规则 / 寒暄不计入 / 建单去重 / 超时关闭）。

### 10.8 下节课预告

**第 38 节：多模态交互系统全流程开发与联调优化**
- Vue 3 三端界面（用户端 / 坐席端 / 管理后台）与路由权限；
- WebSocket 实时链路：断线重连、同账号多端互踢（4409）、消息一致性；
- 全系统联调、异常兜底与体验优化。

---

## 📚 参考资料

- 课件：`第37节-多模态与多智能体协同开发实现.pptx`
- 代码：`core/ai/crew_agents.py` —— CrewAI 三角色编排与流式
- 代码：`core/ai/crew_tools.py` —— 进程内工具与 MCP 工具工厂
- 代码：`core/service/handoff.py` —— 转人工判定 / 摘要 / 工单分类
- 代码：`core/service/session_flow.py` —— 会话回合 Flow 编排
- 代码：`core/service/session_router.py` —— 会话生命周期与坐席动作
- 代码：`core/service/turn_context.py` —— 单轮记录器
- 代码：`core/infra/mcp_client.py` / `mcp_bridge.py` —— MCP 客户端与常驻连接桥
- 代码：`mcp_servers/it_service/server.py` —— 工单 MCP Server（8 工具）
- 测试：`tests/test_handoff.py`

> 🎉 恭喜你完成了本节课的学习！
> 现在你不仅知道多智能体协同怎么写，更知道
> **为什么要这样分工、为什么把规则排成这个顺序、为什么要跨线程搭一座桥**。